# Q08: shifted-validation proxy (evaluation only)

NYU Tandon Deep Learning (Fall 2026), Project 1 ([Kaggle competition](https://www.kaggle.com/competitions/cifar-10-res-net-architecture-challenge-dl)).
Clean internal validation (96–97%) and the reported public scores (0.863–0.889) rank models differently, and the
test images show modest appearance differences ([data_analysis.md](data_analysis.md#interpretation)); no measured
cause of the gap is established. This notebook adds a **generic, fixed** shifted copy of the existing 5k validation
split, so models and recipes can be compared on clean validation **and** on robustness to common photometric and
degradation shifts. It is a diagnostic, not an independent holdout and not a known predictor of hidden-test
accuracy. It trains nothing and never reads test images.

| | |
|---|---|
| Part 1 (section 4) | Deterministically corrupts the raw uint8 validation images of the training notebooks' exact split (`stratified_split`, `VAL_SIZE=5000`, `SPLIT_SEED=0`, cache `42749a486a13`): 13 corruptions × 3 severities = 39 sets, version `Q08_SET_VERSION=1`. SHA-256 fingerprints of every array and of the whole set go into a reference manifest. |
| Part 2 (section 6) | Rebuilds any A/B/C/E/F/G checkpoint from its saved architecture with **verbatim copies** of the six model definitions (section 5), checks dataset identity, split, parameter count and that clean validation reproduces the stored accuracy, then evaluates best/last × raw/EMA × plain/flip on clean plus all 39 sets. |
| Severity values | CIFAR-10-C (Hendrycks & Dietterich, 2019, `make_cifar_c.py`) severities where that reference has the corruption; mirrored magnitudes, defined a priori, for the directions it lacks. **Not fitted to the hidden-test pixel statistics or tuned on public scores.** |
| Section 8 (optional) | Ensemble diversity of the evaluated runs from their saved per-image probabilities: pairwise disagreement (clean and per Q08 family), averaged-probability ensemble accuracy, and a greedy forward selection on clean validation with the Q08 score alongside (teacher selection for distillation, T05). |
| Headline metric | `q08_score_pct`: mean accuracy of the four families (photometric, sharpness, noise, compression), each family weighted equally; also clean accuracy, every corruption × severity, the plain mean over all 39 sets, per-family means, the worst family, per-class recall changes and drops versus clean. |

**Usage on Colab.** Upload this notebook, select an **A100 GPU**, keep the competition data/`cache.zip` in
`MyDrive/DL_Project1/` as for training, list the run IDs in `Q08_RUNS` (section 1) and *Run all*. Checkpoints are
read from `MyDrive/DL_Project1/result/checkpoints/<RUN_ID>/` and then from each `Q08_EXTRA_RESULT_ROOTS` folder
(default: H.W.'s shared folder as the MyDrive shortcut `HW_DL_Project1`, i.e.
`/content/drive/MyDrive/HW_DL_Project1/result`), read-only; nothing in a run's training artifacts is modified and
every output goes to this account's `result/`. Each report records the folder its checkpoints came from.
Generating the set took about 1.2 s per 1,000 images on a laptop CPU (about 6 s for the 5k split). Evaluation time on
an A100 has not been measured yet; per run it covers best/last × raw/EMA × two flips of 40 × 5,000 images.

**Outputs.** `result/manifests/Q08_shiftval-v1_set-manifest.json` (reference fingerprints, written on the first full
generation), per run `result/evaluations/validation/<RUN_ID>/<MM-DD-HH-mm>_<id>_q08-shiftval-v1.json/.csv/.npz` (metrics for every
protocol, a long-format table, and float16 per-image probabilities of the primary and submission protocols), and
`result/reports/<MM-DD-HH-mm>_Q08_shiftval-v1_summary.csv`, `..._severity-check.csv` and `..._ensemble-diversity.json`.

**Sharing the set.** By default every teammate regenerates the set from the competition cache and the notebook
checks it against the reference manifest. Alternatively set `Q08_EXPORT_NPZ=True` once: the whole set (39 corrupted
copies, clean images, labels, indices and manifest) is written as one compressed `.npz` to
`MyDrive/DL_Project1/q08_shiftval/Q08_shiftval-v1_set.npz`, deliberately **outside** `result/` and never in Git
because it contains dataset-derived images. Share that file manually; a notebook that finds it there loads it
(`Q08_SOURCE="auto"`) and verifies every array.

**Local smoke test (CPU/MPS).** Set `Q08_SMOKE_N=200` (and paths if needed): only the first 200 validation images are
used, results are marked `-smoke`, and the reference manifest is neither written nor checked.

## 1. Configuration

Edit this cell, then Run all. `Q08_RUNS` lists exact run IDs (legacy IDs are translated as in the training
notebooks); they are found in `RESULT_DIR` first, then in the read-only `Q08_EXTRA_RESULT_ROOTS` (another
account's result folder through a MyDrive shortcut; on Colab a missing shortcut is awaited for
`Q08_EXTRA_ROOT_WAIT_S` seconds, then skipped with a warning). The default list holds every reviewed run except older
runs whose architecture equals a newer run that only trained longer (F1, A5/T26) and the retired C1; C2 stays
deliberately as the matched 200-epoch control of C9/T57. Every listed run is evaluated with its saved **best** and **last** checkpoints, raw and EMA weights,
with and without flip TTA. The **pre-declared primary protocol** is the last checkpoint with EMA weights (raw
weights when a run has no EMA) and flip TTA, so runs are compared without per-run best-epoch selection; the
protocol of each run's saved submission CSV is flagged too.

`VAL_SIZE`, `SPLIT_SEED` and `Q08_COMPETITION_CHECKS` must stay as given: the notebook refuses any other dataset or
split. `Q08_LOCKED_SET_SHA256` holds the set fingerprint, locked on 2026-10-31 after the one-time, report-only
severity check (section 7), so that any regeneration, on any account, must reproduce the locked set.

In [ ]:
# Q08 shifted-validation proxy: evaluation only, no training. Edit the values below, then Run all.

# Part 1: shifted copy of the existing 5k validation split (section 4)
Q08_SOURCE = "auto"          # "auto": load the shared .npz if present, else regenerate | "regenerate" | "npz"
Q08_EXPORT_NPZ = False       # True: also write the whole set as one compressed .npz for manual sharing
Q08_DIR = None               # .npz folder; None: Colab MyDrive/DL_Project1/q08_shiftval, local project1/downloads/q08_shiftval
Q08_LOCKED_SET_SHA256 = "3f12d750b28139569b8912c175dce67dfaadaa471afc838d9f4af62538b797d1"  # locked 2026-10-31 after the report-only severity check
Q08_REQUIRE_EXACT = False    # True: Pillow-dependent arrays (JPEG, pixelate) must also match bit for bit
Q08_SMOKE_N = 0              # > 0: first N validation images only (quick local check; never written as the reference)

# Part 2: saved checkpoints to evaluate (exact run IDs; any of A/B/C/E/F/G). Each run's checkpoints and its
# configs/evaluations/reports are looked up in RESULT_DIR first, then in each Q08_EXTRA_RESULT_ROOTS folder; the
# folder used is recorded in every report. Outputs are written under RESULT_DIR only (extra roots are read-only).
# Legacy Drive folder/file names (e.g. ...-ver-1-grouped-ema) resolve automatically. A run found in no root is
# skipped with a message. Default list: every reviewed run except older runs whose architecture is identical to a
# newer run that only trained longer (F1 -> F2; A5/T26 -> its continuation A6/T59) and the retired C1.
# Checkpoints: A/B/C on H.W.'s Drive (extra root), E/F/G on J.H.'s Drive (RESULT_DIR on J.H.'s Colab).
Q08_RUNS = [
    "10-30-08-40_A_resnet-ver-1-T00-cutout-ema",        # A1 baseline; Kaggle 0.872
    "10-30-20-03_A_resnet-ver-2-T48-eca-ema",           # A2/T48 ECA
    "10-30-21-17_A_resnet-ver-3-T49-mixup-ema",         # A3/T49 Mixup
    "10-30-23-09_A_resnet-ver-4-T22-early-ema",         # A4/T22 extra early block
    "10-31-03-09_A_resnet-ver-6-T59-mid-continue",      # A6/T59 = A5/T26 (mid; Kaggle 0.863) continued to 400 epochs
    "10-31-14-09_A_resnet-ver-7-T14-seed1-ema",         # A7/T14 A1 recipe, seed 1
    "10-31-14-56_A_resnet-ver-8-T21-early-eca",         # A8/T21 early block + ECA
    "10-30-09-44_B_wrn-ver-1-T00-cutout-ema",           # B1 baseline; Kaggle 0.871
    "10-30-19-41_B_wrn-ver-2-T08-wide-ema",             # B2/T08 wide68
    "10-30-21-18_B_wrn-ver-3-T50-mixup-ema",            # B3/T50 Mixup
    "10-30-23-10_B_wrn-ver-4-T24-deep-ema",             # B4/T24 deep56
    "10-31-00-42_B_wrn-ver-5-T23-balanced-ema",         # B5/T23 balanced
    "10-31-14-34_B_wrn-ver-6-T12-seed1-ema",            # B6/T12 B1 recipe, seed 1
    "10-31-15-31_B_wrn-ver-7-T25-shallow-ema",          # B7/T25 WRN-16-5 (shallow80)
    # C2 is kept on purpose although C7/T60 continues it: it is the matched 200-epoch control for C9/T57 (IBN).
    "10-30-09-50_C_seresnet-ver-2-T00-cutout-ema",      # C2 baseline; Kaggle 0.865; severity reference (section 7)
    "10-30-20-04_C_seresnet-ver-3-T06-se8-ema",         # C3/T06 SE r=8
    "10-30-21-47_C_seresnet-ver-4-T51-mixup-ema",       # C4/T51 Mixup
    "10-30-23-43_C_seresnet-ver-5-T01-autoaug-cutout",  # C5/T01; AutoAugment overlaps photometric/sharpness
    "10-31-00-50_C_seresnet-ver-6-T31-reflect-ema",     # C6/T31 reflection padding
    "10-31-03-41_C_seresnet-ver-7-T60-cutout-continue", # C7/T60 = C2 continued to 400 epochs
    "10-31-14-10_C_seresnet-ver-8-T02-autoaug-only",    # C8/T02; AutoAugment overlaps photometric/sharpness
    "10-31-15-18_C_seresnet-ver-9-T57-ibn-ema",         # C9/T57 IBN-a in stage 1 (IBN_STAGE1=True)
    "10-30-21-05_E_pyramid-ver-1-T18-basic110-ema",     # E1/T18; Kaggle 0.889
    "10-30-22-34_F_shake-ver-2-T35-shake40-1800ep",     # F2/T35 1800 epochs; Kaggle 0.873; Drive keeps 10-30-22-34_F_shake-ver-2-shake40-1800ep
    "10-30-21-07_G_resnext-ver-1-T42-grouped-ema",      # G1/T42; Kaggle 0.872; local legacy folder ...-ver-1-grouped-ema
]
# Read-only result folders of other accounts, searched after RESULT_DIR. Colab: H.W.'s shared DL_Project1 folder,
# added to MyDrive as a shortcut named HW_DL_Project1. A missing folder is skipped with a warning.
Q08_EXTRA_RESULT_ROOTS = ["/content/drive/MyDrive/HW_DL_Project1/result"]
Q08_EXTRA_ROOT_WAIT_S = 60          # Colab only: a Drive shortcut can appear a little after mounting; wait this long
Q08_CHECKPOINTS = ("best", "last")  # saved checkpoint roles; "last" is the pre-declared final-epoch rule
Q08_WEIGHTS = ("model", "ema")      # raw and EMA weights; EMA is skipped for a run without it
Q08_REPRO_TOLERANCE_PP = 0.10       # clean accuracy must reproduce the stored values within this many points
Q08_SEVERITY_REFERENCE_RUN = "10-30-09-50_C_seresnet-ver-2-T00-cutout-ema"  # pre-declared (C2); None = first evaluated
Q08_SAVE_ALL_PROBABILITIES = False  # per-image probabilities: False = primary + submission protocols only; True = all 8

# Optional section 8: ensemble diversity (teacher selection for distillation; validation images only)
Q08_ENSEMBLE_RUNS = "evaluated"    # "evaluated": the runs above | list of run IDs (reuses their saved Q08 .npz) | None: skip
Q08_ENSEMBLE_PROTOCOL = "primary"  # "primary" | "submission" | another key such as "best_model_flip_tta" (needs all saved)

# Fixed split and data identity: keep these values (checked against the training notebooks' split)
VAL_SIZE = 5000
SPLIT_SEED = 0
Q08_COMPETITION_CHECKS = True  # require the competition cache fingerprint and the known split hashes

# Hardware (None = automatic). Evaluation runs in fp32 like the training notebooks' validation/prediction.
DEVICE = None            # "cuda" | "mps" | "cpu"; None = $CIFAR_DEVICE, else cuda > mps > cpu
AMP = "off"              # not used for evaluation; kept for the shared environment cell
CHANNELS_LAST = None
CUDNN_BENCHMARK = None
PIN_MEMORY = None
WORKERS = None
EVAL_BATCH_SIZE = 1000   # same batch as the training notebooks' validation and prediction

# Runtime host, independent of the editor: auto | colab | kaggle | local.
# For a VS Code kernel connected to Colab, set "colab" if auto detection is ambiguous.
RUNTIME_ENV = "auto"     # explicit value overrides CIFAR_ENV; auto uses CIFAR_ENV then detection

# Paths (None = $CIFAR_DATA_DIR / $CIFAR_CACHE_DIR / $CIFAR_RESULT_DIR, else the default of the environment)
AUTO_DOWNLOAD = False      # Kaggle download/login code is commented out; use uploaded data
SAVE_CACHE_TO_DRIVE = True # on Colab, automatically save/verify MyDrive/DL_Project1/cache.zip
DATA_DIR = None            # Colab: MyDrive/DL_Project1/data/ (folder or override with a ZIP path)
CACHE_DIR = None
RESULT_DIR = None          # checkpoints are read from RESULT_DIR/checkpoints; reports are written beside them
SHOW_SUMMARY = False       # the copied model cells print no layer table here

## 2. Environment, paths and device

The two code cells below are verbatim copies of the training notebooks' shared environment cells (the regression
tests check this), so detection, Drive mounting and paths behave exactly as in training: Colab reads and writes
`MyDrive/DL_Project1/result` and keeps the cache in `/content/cache`; local runs use `project1/data`, `project1/cache`
and `project1/result`. On Colab select an A100 GPU. Evaluation runs in fp32 (no autocast), as in the training
notebooks' validation and prediction.

In [ ]:
import copy
import csv
import hashlib
import importlib.util
import importlib.metadata
import io
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import tempfile
import zlib
import zipfile
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from matplotlib.ticker import MaxNLocator
from PIL import Image

ENVS = ("local", "colab", "kaggle")
KAGGLE_SLUG = "cifar-10-res-net-architecture-challenge-dl"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/DL_Project1"
WORK_DIRS = {"colab": "/content", "kaggle": "/kaggle/working"}
COLAB_CKPT_EVERY = 5  # Drive writes are slow; best-ckpt.pt and the final epoch are always saved
PROJECT_DIR = Path.cwd().resolve()
LOCAL_PROJECT_DIR = PROJECT_DIR.parent if PROJECT_DIR.name == "Archive" else PROJECT_DIR
# Start method for the AutoAugment DataLoader workers: "fork" lets them use the Dataset class defined in this
# notebook. Under "spawn" (macOS, Windows) they cannot import it, so the pipeline runs in the main process there.
MP_CONTEXT = "fork" if sys.platform.startswith("linux") else None

_warned = set()


def _getenv(name):
    value = os.environ.get(name, "").strip()
    return value or None


def warn_once(msg):
    if msg not in _warned:
        _warned.add(msg)
        print(f"Warning: {msg}", flush=True)


def _module_exists(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False


def detect_env():
    """Detect the kernel host, independent of the notebook editor (browser or VS Code)."""
    configured = str(globals().get("RUNTIME_ENV") or "auto").strip().lower()
    if configured not in ("auto",) + ENVS:
        raise ValueError(f"RUNTIME_ENV={configured!r}; expected auto or one of {ENVS}")
    requested = configured if configured != "auto" else (_getenv("CIFAR_ENV") or "auto").lower()
    if requested not in ("auto",) + ENVS:
        raise ValueError(f"CIFAR_ENV={requested!r}; expected auto or one of {ENVS}")
    if requested != "auto":
        return requested
    # Runtime markers beat directory names: /kaggle/input may also exist on Colab.
    if any(_getenv(key) for key in ("COLAB_RELEASE_TAG", "COLAB_BACKEND_VERSION")):
        return "colab"
    if _getenv("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    if _module_exists("google.colab"):
        return "colab"
    # A directory alone is weak evidence; use the full layout only as a fallback.
    if os.path.isdir("/kaggle/input") and os.path.isdir("/kaggle/working"):
        return "kaggle"
    return "local"


def work_dir(env=None):
    env = env or detect_env()
    return Path(_getenv("CIFAR_WORK_DIR") or WORK_DIRS.get(env, PROJECT_DIR))


def drive_dir():
    return Path(_getenv("CIFAR_DRIVE_DIR") or DRIVE_PROJECT_DIR)


def drive_mounted():
    return drive_dir().parent.is_dir()  # .../MyDrive exists once Drive is mounted


def kaggle_input_dir():
    return Path(_getenv("CIFAR_KAGGLE_INPUT") or "/kaggle/input")


def find_competition_data(root=None, max_depth=6):
    """Directories under `root` that contain train_labels.csv, best match first.

    Breadth-first, follows symlinks, skips *_images folders (50k files) and hidden folders.
    """
    root = Path(root or kaggle_input_dir())
    found, frontier = [], [(root, 0)]
    while frontier:
        d, depth = frontier.pop(0)
        try:
            entries = sorted(os.scandir(d), key=lambda e: e.name)
        except OSError:
            continue
        if any(e.name == "train_labels.csv" and e.is_file() for e in entries):
            found.append(d)
        if depth < max_depth:
            frontier += [(Path(e.path), depth + 1) for e in entries
                         if e.is_dir() and not e.name.endswith("_images") and not e.name.startswith(".")]

    def rank(d):
        complete = (d / "test.csv").is_file() and (d / "sample_submission.csv").is_file()
        return (not complete, KAGGLE_SLUG not in str(d), len(d.parts), str(d))

    return sorted(found, key=rank)


@dataclass
class Paths:
    env: str
    data_dir: Path
    cache_dir: Path
    result_dir: Path


def resolve_paths(data_dir=None, cache_dir=None, result_dir=None, env=None):
    """Data, cache and runs directories: argument > environment variable > environment default."""
    env = env or detect_env()
    data = data_dir or _getenv("CIFAR_DATA_DIR")
    cache = cache_dir or _getenv("CIFAR_CACHE_DIR")
    runs = result_dir or _getenv("CIFAR_RESULT_DIR")
    if env == "local":
        data = data or LOCAL_PROJECT_DIR / "data"
        cache = cache or LOCAL_PROJECT_DIR / "cache"
        runs = runs or LOCAL_PROJECT_DIR / "result"
    elif env == "colab":
        cache = cache or work_dir(env) / "cache"
        data = data or drive_dir() / "data"  # manually uploaded ZIPs or extracted competition files
        if runs is None:
            if drive_mounted():
                runs = drive_dir() / "result"
            else:
                runs = work_dir(env) / "result"
                warn_once(f"Google Drive is not mounted ({drive_dir().parent} missing); writing results to "
                          f"{runs}, which is lost when the runtime is recycled")
    else:  # kaggle
        cache = cache or work_dir(env) / "cache"
        runs = runs or work_dir(env) / "result"
        if data is None:
            found = find_competition_data()
            if found:
                data = found[0]
            else:
                data = work_dir(env) / "data"
                warn_once(f"no train_labels.csv found under {kaggle_input_dir()}; checking cache or manually supplied data")
    return Paths(env, Path(data), Path(cache), Path(runs))


def default_ckpt_every(env=None):
    return COLAB_CKPT_EVERY if (env or detect_env()) == "colab" else 1


def available_cpus():
    try:
        return len(os.sched_getaffinity(0))
    except AttributeError:  # macOS
        return os.cpu_count() or 1


def default_workers(device_type):
    """DataLoader workers (used by the per-image AutoAugment pipeline only)."""
    n = available_cpus()
    if device_type == "cuda":
        return min(8, max(2, n - 1))
    return min(4, max(1, n // 2))  # 4 on the 12-core M4 Pro, as before


def get_device(name=None):
    """`name` (DEVICE), else $CIFAR_DEVICE, else cuda > mps > cpu."""
    name = name or _getenv("CIFAR_DEVICE") or "auto"
    if name == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        if torch.backends.mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    return torch.device(name)


def cuda_bf16_native(device):
    return torch.cuda.get_device_capability(device) >= (8, 0)  # Ampere or newer (L4, A100, ...)


def resolve_amp(requested, device):
    """'bf16', 'fp16' or 'off'. requested: auto | on | off | bf16 | fp16 (bools accepted)."""
    req = {True: "on", False: "off", None: "auto"}.get(requested, requested)
    req = str(req).lower()
    if req == "off":
        return "off"
    if device.type != "cuda":
        if req != "auto":
            # MPS: fp16 autocast gave no speed-up on an M4 Pro, and torch 2.5.1 GradScaler fails on MPS.
            print(f"Warning: AMP is only used on CUDA; training in fp32 on {device.type}.")
        return "off"
    if req == "fp16":
        return "fp16"
    if cuda_bf16_native(device):
        return "bf16"
    if req == "bf16":
        print("Warning: this GPU has no native bf16; using fp16 autocast + GradScaler instead.")
    return "fp16"


# torch.compile modes for the training step (README "Compilation"); a speed option, never part of the recipe.
COMPILE_MODES = ("off", "default", "reduce-overhead", "max-autotune", "max-autotune-no-cudagraphs")


def resolve_compile(requested, device, name="COMPILE"):
    """One of COMPILE_MODES. Compilation is used on CUDA only; elsewhere it warns and runs uncompiled ("off")."""
    if not isinstance(requested, str) or requested not in COMPILE_MODES:
        raise ValueError(f"{name} = {requested!r}; choose one of {COMPILE_MODES}")
    if requested != "off" and device.type != "cuda":
        print(f"Warning: {name}={requested!r} is only used on CUDA; running uncompiled on {device.type}.")
        return "off"
    return requested


def resolve_hardware(args, device):
    """Fill in the automatic (None) speed settings on `args` for `device`, in place."""
    cuda = device.type == "cuda"
    args.amp = resolve_amp(getattr(args, "amp", "auto"), device)
    args.compile = resolve_compile(getattr(args, "compile", "off"), device)
    if args.channels_last is None:
        args.channels_last = cuda
    elif args.channels_last and device.type == "mps":
        # torch 2.5.1 MPS: backward through conv/BN/ReLU in channels_last raises a stride error.
        print("Warning: CHANNELS_LAST is not supported on MPS with this PyTorch; disabling it.")
        args.channels_last = False
    if getattr(args, "cudnn_benchmark", None) is None:
        args.cudnn_benchmark = cuda
    if getattr(args, "pin_memory", None) is None:
        args.pin_memory = cuda
    if getattr(args, "workers", None) is None:
        args.workers = default_workers(device.type)
    if MP_CONTEXT is None:
        args.workers = 0  # see MP_CONTEXT
    return args


def device_name(device):
    if device.type == "cuda":
        props = torch.cuda.get_device_properties(device)
        return f"cuda ({props.name}, {props.total_memory / 2**30:.0f} GB)"
    if device.type == "cpu":
        return f"cpu ({available_cpus()} cores)"
    return f"{device.type} ({platform.machine()})"


def _on(x):
    return "on" if x else "off"


def summary_line(paths, device=None, args=None):
    """One line: environment, device, torch, speed settings and the output directories."""
    import torch

    parts = [f"env {paths.env}"]
    if device is not None:
        parts.append(device_name(device))
    parts.append(f"torch {torch.__version__}")
    if args is not None:
        parts.append(f"amp {args.amp} | channels_last {_on(args.channels_last)} | "
                     f"cudnn.benchmark {_on(args.cudnn_benchmark)} | workers {args.workers} | "
                     f"pin_memory {_on(args.pin_memory)}")
        if getattr(args, "compile", "off") != "off":
            parts.append(f"compile {args.compile}")
    parts.append(f"data {paths.data_dir} | cache {paths.cache_dir} | result {paths.result_dir}")
    return "[env] " + " | ".join(parts)

def run_timestamp():
    return datetime.now(ZoneInfo("America/New_York")).strftime("%m-%d-%H-%M")


# Exact historical aliases: checkpoint bytes and signed export manifests remain unchanged.
RUN_ID_ALIASES = {'10-30-08-40_A_resnet-ver-1-cutout-ema': '10-30-08-40_A_resnet-ver-1-T00-cutout-ema', '10-30-09-44_B_wrn-ver-1-cutout-ema': '10-30-09-44_B_wrn-ver-1-T00-cutout-ema', '10-30-09-46_C_seresnet-ver-1-cutout-ema': '10-30-09-46_C_seresnet-ver-1-T00-cutout-ema', '10-30-09-50_C_seresnet-ver-2-cutout-ema': '10-30-09-50_C_seresnet-ver-2-T00-cutout-ema', '10-30-19-41_B_wrn-ver-2-wide-ema': '10-30-19-41_B_wrn-ver-2-T08-wide-ema', '10-30-20-03_A_resnet-ver-6-eca-ema': '10-30-20-03_A_resnet-ver-2-T48-eca-ema', '10-30-20-04_C_seresnet-ver-8-se8-ema': '10-30-20-04_C_seresnet-ver-3-T06-se8-ema', '10-30-21-05_E_pyramid-ver-1-basic110-ema': '10-30-21-05_E_pyramid-ver-1-T18-basic110-ema', '10-30-21-07_G_resnext-ver-1-grouped-ema': '10-30-21-07_G_resnext-ver-1-T42-grouped-ema', '10-30-21-17_A_resnet-ver-7-mixup-ema': '10-30-21-17_A_resnet-ver-3-T49-mixup-ema', '10-30-21-18_B_wrn-ver-8-mixup-ema': '10-30-21-18_B_wrn-ver-3-T50-mixup-ema', '10-30-21-42_F_shake-ver-1-shake40': '10-30-21-42_F_shake-ver-1-T35-shake40', '10-30-21-47_C_seresnet-ver-31-mixup-ema': '10-30-21-47_C_seresnet-ver-4-T51-mixup-ema', '10-30-23-09_A_resnet-ver-4-early-ema': '10-30-23-09_A_resnet-ver-4-T22-early-ema', '10-30-23-10_B_wrn-ver-6-deep-ema': '10-30-23-10_B_wrn-ver-4-T24-deep-ema', '10-30-23-43_C_seresnet-ver-3-autoaug-cutout': '10-30-23-43_C_seresnet-ver-5-T01-autoaug-cutout', '10-30-23-56_A_resnet-ver-5-mid-ema': '10-30-23-56_A_resnet-ver-5-T26-mid-ema', '10-31-00-42_B_wrn-ver-5-balanced-ema': '10-31-00-42_B_wrn-ver-5-T23-balanced-ema', '10-31-00-50_C_seresnet-ver-19-reflect-ema': '10-31-00-50_C_seresnet-ver-6-T31-reflect-ema', '10-30-22-34_F_shake-ver-2-shake40-1800ep': '10-30-22-34_F_shake-ver-2-T35-shake40-1800ep', '10-31-20-06_E_pyramid-ver-3-T61-autoaug-only': '10-31-20-06_E_pyramid-ver-3-T69-autoaug-only'}


def canonical_run_id(run_id):
    return RUN_ID_ALIASES.get(run_id, run_id)


def run_task_id(run_id):
    match = re.search(r"-ver-\d+-(T\d{2,})-", canonical_run_id(run_id))
    return match.group(1) if match else "T00"


def run_paths(result_root, run_id, create=False):
    """One timestamped ID across directories grouped by artifact type."""
    if not run_id or Path(run_id).name != run_id or run_id in (".", ".."):
        raise ValueError("Run ID must be a single directory name")
    run_id = canonical_run_id(run_id)
    root = Path(result_root)
    folders = {"checkpoints": "checkpoints", "configs": "configs", "logs": "logs",
               "validation": "evaluations/validation", "test": "evaluations/test",
               "plots": "plots", "progress": "progress", "manifests": "manifests", "exports": "exports"}
    paths = SimpleNamespace(root=root, run_id=run_id,
                            **{key: root / folder / run_id for key, folder in folders.items()})
    if not paths.checkpoints.exists():
        for legacy, canonical in RUN_ID_ALIASES.items():
            old_dir = root / "checkpoints" / legacy
            if canonical == run_id and old_dir.is_dir():
                paths.checkpoints = old_dir
                break
    if create:
        for key in folders:
            getattr(paths, key).mkdir(parents=True, exist_ok=True)
    return paths


def paths_for_run(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    if checkpoint_dir.parent.name != "checkpoints":
        raise ValueError("Expected result/checkpoints/<RUN_ID>; legacy flat runs must be migrated explicitly")
    return run_paths(checkpoint_dir.parent.parent, checkpoint_dir.name)


def artifact_path(artifacts, category, role, extension, event=False, unique=False):
    """Stable training paths; dated evaluation/export paths with collision-safe role suffixes."""
    folder = getattr(artifacts, category)
    prefix = artifacts.run_id
    if event:
        identifier = artifacts.run_id.split('_', 1)[1] if '_' in artifacts.run_id else artifacts.run_id
        prefix = f"{run_timestamp()}_{identifier}"
    stem = f"{prefix}_{role}"
    path = folder / (stem + extension)
    if unique:
        index = 1
        while any(folder.glob(path.stem + '*')):
            index += 1
            path = folder / (f"{stem}-{index}" + extension)
    return path


def checkpoint_path(run_dir, which):
    if which not in ('best', 'last'):
        raise ValueError("Checkpoint role must be best or last")
    run_dir = Path(run_dir)
    canonical = canonical_run_id(run_dir.name)
    candidates = [run_dir.parent / canonical / f"{canonical}_{which}-ckpt.pt",
                  run_dir / f"{run_dir.name}_{which}-ckpt.pt"]
    for legacy, target in RUN_ID_ALIASES.items():
        if target == canonical:
            candidates.extend([run_dir.parent / canonical / f"{legacy}_{which}-ckpt.pt",
                               run_dir.parent / legacy / f"{legacy}_{which}-ckpt.pt"])
    return next((p for p in candidates if p.is_file()), candidates[0])


In [ ]:
ENV = detect_env()
print(f"[runtime] detected {ENV} | kernel {sys.executable} | cwd {Path.cwd()} | "
      f"RUNTIME_ENV={RUNTIME_ENV!r} | CIFAR_ENV={_getenv('CIFAR_ENV')!r}")
if ENV == "colab":
    try:
        from google.colab import drive
    except ImportError:  # CIFAR_ENV=colab outside Google Colab (testing)
        print("google.colab is not available: skipping drive.mount")
    else:
        drive.mount("/content/drive")

PATHS = resolve_paths(DATA_DIR, CACHE_DIR, RESULT_DIR, ENV)
DEV = get_device(DEVICE)
if ENV == "colab" and DEV.type != "cuda" and (DEVICE or _getenv("CIFAR_DEVICE")) != "cpu":
    raise RuntimeError("No GPU: choose Runtime > Change runtime type > A100 GPU, then Runtime > Run all again")
HW = resolve_hardware(SimpleNamespace(amp=AMP, channels_last=CHANNELS_LAST, cudnn_benchmark=CUDNN_BENCHMARK,
                                      pin_memory=PIN_MEMORY, workers=WORKERS), DEV)
print(summary_line(PATHS, DEV, HW))
print(f"python {platform.python_version()} | torchvision {torchvision.__version__} | numpy {np.__version__} | "
      f"{platform.platform()} | {available_cpus()} CPUs")
if ENV == "colab":
    print(f"Google Drive folder {drive_dir()}: {'found' if drive_dir().is_dir() else 'NOT FOUND'}")
if ENV == "kaggle":
    print(f"Kaggle input {kaggle_input_dir()}: train_labels.csv in "
          f"{[str(d) for d in find_competition_data()] or 'none'}")
if DEV.type == "cuda" and shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 3. Data and the shared validation split

The data helpers and the data cell are verbatim copies of the training notebooks' cells: the notebook reuses a
verified local cache, then a compatible Drive `cache.zip`, then the manually uploaded competition data. The next
cell holds verbatim copies of the split, inference and checkpoint helpers it needs. The split cell then recomputes
the 45k/5k split with the training notebooks' `stratified_split` and checks the cache fingerprint
(`42749a486a13…`) and the SHA-256 of the validation indices and labels against the values shared by every saved
validation export of the A/B/C/E/F/G runs. Corruptions are applied to these **raw uint8 images, before**
mean/std normalization; each checkpoint's own saved normalization is applied at evaluation, exactly as in training.

In [ ]:
NUM_CLASSES = 10
CLASSES = ("airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck")
CACHE_FORMAT = 2
SOURCE_CSVS = ("train_labels.csv", "test.csv", "sample_submission.csv")
CACHE_ARRAYS = ("train_images.npy", "train_labels.npy", "train_ids.npy", "test_images.npy", "test_ids.npy")


def sha256_file(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _decode_pngs(paths):
    def load(p):
        with Image.open(p) as im:
            return np.asarray(im.convert("RGB"), dtype=np.uint8)

    with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 1)) as pool:
        return np.stack(list(pool.map(load, paths, chunksize=256)))


def _save_npy(path, arr):
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        np.save(f, arr)
    os.replace(tmp, path)


def _copy_csvs(data_dir, cache_dir):
    if data_dir.resolve() == cache_dir.resolve():
        return
    for name in SOURCE_CSVS:
        tmp = cache_dir / (name + ".tmp")
        shutil.copyfile(data_dir / name, tmp)
        os.replace(tmp, cache_dir / name)


def _write_meta(cache_dir, n_train, n_test, shape, note):
    files = {name: {"sha256": sha256_file(cache_dir / name), "bytes": (cache_dir / name).stat().st_size}
             for name in CACHE_ARRAYS + SOURCE_CSVS}
    fingerprint = hashlib.sha256("".join(f"{n} {files[n]['sha256']}\n" for n in sorted(files)).encode())
    meta = {"format": CACHE_FORMAT, "n_train": n_train, "n_test": n_test, "shape": list(shape),
            "fingerprint": fingerprint.hexdigest(), "files": files, "note": note,
            "written": datetime.now().isoformat(timespec="seconds")}
    tmp = cache_dir / "meta.json.tmp"
    tmp.write_text(json.dumps(meta, indent=2))
    os.replace(tmp, cache_dir / "meta.json")  # written last: a cache without it is incomplete
    return meta


def read_meta(cache_dir):
    path = Path(cache_dir) / "meta.json"
    return json.loads(path.read_text()) if path.exists() else None


def build_cache(data_dir, cache_dir):
    """Decode all PNGs once into uint8 arrays (N, 32, 32, 3) and save them with the CSVs and meta.json."""
    data_dir, cache_dir = Path(data_dir), Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)

    train = pd.read_csv(data_dir / "train_labels.csv")
    print(f"[cache] decoding {len(train):,} training PNGs ...", flush=True)
    x = _decode_pngs([data_dir / "train_images" / f for f in train["filename"]])
    _save_npy(cache_dir / "train_images.npy", x)
    _save_npy(cache_dir / "train_labels.npy", train["label"].to_numpy(np.int64))
    _save_npy(cache_dir / "train_ids.npy", train["id"].to_numpy(np.int64))

    test = pd.read_csv(data_dir / "test.csv")
    print(f"[cache] decoding {len(test):,} test PNGs ...", flush=True)
    x = _decode_pngs([data_dir / "test_images" / f for f in test["filename"]])
    _save_npy(cache_dir / "test_images.npy", x)
    _save_npy(cache_dir / "test_ids.npy", test["id"].to_numpy(np.int64))

    _copy_csvs(data_dir, cache_dir)
    validate_cache_arrays(cache_dir)
    meta = _write_meta(cache_dir, len(train), len(test), x.shape[1:], f"built from {data_dir}")
    print(f"[cache] wrote {cache_dir} (fingerprint {meta['fingerprint'][:12]})", flush=True)
    return meta


def _upgrade_format1(data_dir, cache_dir):
    """Add CSV copies and hashes to a format-1 cache if its arrays match the CSVs. Returns success."""
    train = pd.read_csv(data_dir / "train_labels.csv")
    test = pd.read_csv(data_dir / "test.csv")
    try:
        x_train = np.load(cache_dir / "train_images.npy", mmap_mode="r")
        x_test = np.load(cache_dir / "test_images.npy", mmap_mode="r")
        ok = (x_train.shape[0] == len(train) and x_test.shape[0] == len(test)
              and x_train.shape[1:] == x_test.shape[1:]
              and np.array_equal(np.load(cache_dir / "train_labels.npy"), train["label"].to_numpy(np.int64))
              and np.array_equal(np.load(cache_dir / "train_ids.npy"), train["id"].to_numpy(np.int64))
              and np.array_equal(np.load(cache_dir / "test_ids.npy"), test["id"].to_numpy(np.int64)))
    except (OSError, ValueError):
        ok = False
    if not ok:
        return False
    shape = x_train.shape[1:]
    del x_train, x_test
    _copy_csvs(data_dir, cache_dir)
    validate_cache_arrays(cache_dir)
    meta = _write_meta(cache_dir, len(train), len(test), shape, f"format 1 cache upgraded with {data_dir}")
    print(f"[cache] upgraded {cache_dir} to format {CACHE_FORMAT} (arrays unchanged, "
          f"fingerprint {meta['fingerprint'][:12]})", flush=True)
    return True


def _cache_problem(meta, cache_dir, data_dir, has_csvs):
    """None if the format-2 cache is complete and matches the CSVs in data_dir, else the reason."""
    for name, info in meta["files"].items():
        path = cache_dir / name
        if not path.exists() or path.stat().st_size != info["bytes"]:
            return f"{name} is missing or has the wrong size"
    if has_csvs and data_dir.resolve() != cache_dir.resolve():
        for name in SOURCE_CSVS:
            if sha256_file(data_dir / name) != meta["files"][name]["sha256"]:
                return f"{data_dir / name} differs from the cached copy"
    return None


def ensure_cache(data_dir, cache_dir):
    """Return the cache dir, building, upgrading or rebuilding the cache only when needed.

    A complete format-2 cache is used as is; if data_dir also holds the CSVs, they must match the
    cached copies (otherwise the cache is rebuilt from the PNGs).
    """
    data_dir, cache_dir = Path(data_dir), Path(cache_dir)
    has_csvs = all((data_dir / n).is_file() for n in SOURCE_CSVS)
    has_pngs = has_csvs and (data_dir / "train_images").is_dir() and (data_dir / "test_images").is_dir()
    meta = read_meta(cache_dir)
    if meta is not None and meta.get("format") == CACHE_FORMAT:
        problem = _cache_problem(meta, cache_dir, data_dir, has_csvs)
        if problem is None:
            return cache_dir
        print(f"[cache] {problem}; rebuilding", flush=True)
    elif meta is not None and has_csvs and all((cache_dir / n).exists() for n in CACHE_ARRAYS):
        if _upgrade_format1(data_dir, cache_dir):
            return cache_dir
        print("[cache] format 1 cache does not match the CSVs; rebuilding", flush=True)
    if not has_pngs and meta is not None and meta.get("format") is None:
        raise FileNotFoundError(
            f"{cache_dir} is a format-1 cache and {data_dir} has no CSVs to upgrade it with; "
            f"set DATA_DIR to the folder with the competition CSVs")
    if not has_pngs:
        raise FileNotFoundError(
            f"no usable cache in {cache_dir} and no PNG images in {data_dir} to build one; set DATA_DIR "
            f"(folder with train_labels.csv, train_images/, ...) or CACHE_DIR, or on Colab upload cache.zip "
            f"to {drive_dir()}")
    build_cache(data_dir, cache_dir)
    return cache_dir


def verify_cache(cache_dir):
    """Check every cached file against the sha256 in meta.json; return meta (raises on a mismatch)."""
    cache_dir = Path(cache_dir)
    meta = read_meta(cache_dir)
    _check_cache_manifest(meta)
    bad = [n for n, info in meta["files"].items()
           if not (cache_dir / n).exists() or sha256_file(cache_dir / n) != info["sha256"]]
    if bad:
        raise RuntimeError(f"cache files do not match meta.json: {bad}")
    n_train, n_test = validate_cache_arrays(cache_dir)
    if (meta["n_train"], meta["n_test"], meta["shape"]) != (n_train, n_test, [32, 32, 3]):
        raise RuntimeError("Cache metadata does not match its arrays")
    return meta


def validate_cache_arrays(cache_dir):
    """Validate shapes, labels, IDs, and the correspondence between arrays and CSVs."""
    cache_dir = Path(cache_dir)
    frames = {name: pd.read_csv(cache_dir / name) for name in SOURCE_CSVS}
    for name, columns in (("train_labels.csv", ("id", "filename", "label")),
                          ("test.csv", ("id", "filename")),
                          ("sample_submission.csv", ("id", "label"))):
        frame = frames[name]
        if not set(columns).issubset(frame.columns) or frame.empty:
            raise ValueError(f"{name}: missing columns or no rows")
        if not pd.api.types.is_integer_dtype(frame["id"]) or not frame["id"].is_unique:
            raise ValueError(f"{name}: IDs must be unique integers")
        if "filename" in frame and (frame["filename"].isna().any() or not frame["filename"].is_unique):
            raise ValueError(f"{name}: filenames must be present and unique")
    train, test = frames["train_labels.csv"], frames["test.csv"]
    if not pd.api.types.is_integer_dtype(train["label"]) or not train["label"].between(0, 9).all():
        raise ValueError("Training labels must be integers in 0..9")
    for split, frame in (("train", train), ("test", test)):
        images = np.load(cache_dir / f"{split}_images.npy", mmap_mode="r", allow_pickle=False)
        ids = np.load(cache_dir / f"{split}_ids.npy", allow_pickle=False)
        if images.dtype != np.uint8 or images.shape != (len(frame), 32, 32, 3):
            raise ValueError(f"{split} images must be uint8 NHWC with shape (N,32,32,3)")
        if not np.array_equal(ids, frame["id"].to_numpy()):
            raise ValueError(f"{split} array IDs differ from CSV order")
    labels = np.load(cache_dir / "train_labels.npy", allow_pickle=False)
    if not np.issubdtype(labels.dtype, np.integer) or not np.array_equal(labels, train["label"].to_numpy()):
        raise ValueError("Cached labels differ from training CSV")
    if set(frames["sample_submission.csv"]["id"]) != set(test["id"]):
        raise ValueError("Sample submission and test IDs differ")
    return len(train), len(test)


def cache_fingerprint(cache_dir):
    meta = read_meta(cache_dir) or {}
    return meta.get("fingerprint")


def load_train(data_dir, cache_dir, mmap=False):
    cache_dir = ensure_cache(data_dir, cache_dir)
    images = np.load(cache_dir / "train_images.npy", mmap_mode="r" if mmap else None)
    return images, np.load(cache_dir / "train_labels.npy"), np.load(cache_dir / "train_ids.npy")


def load_test(data_dir, cache_dir):
    cache_dir = ensure_cache(data_dir, cache_dir)
    return np.load(cache_dir / "test_images.npy"), np.load(cache_dir / "test_ids.npy")


def _cache_matches_source(meta, source_dir):
    """An explicitly supplied set of CSVs must agree with a reused cache."""
    source_dir = Path(source_dir)
    return all(not (source_dir / name).is_file() or
               sha256_file(source_dir / name) == meta['files'][name]['sha256'] for name in SOURCE_CSVS)


def _check_cache_manifest(meta):
    required = set(CACHE_ARRAYS + SOURCE_CSVS)
    if (not isinstance(meta, dict) or meta.get('format') != CACHE_FORMAT
            or not isinstance(meta.get('files'), dict) or set(meta['files']) != required):
        raise ValueError('Cache needs a format-2 manifest with all required files')
    for info in meta['files'].values():
        if (not isinstance(info, dict) or not isinstance(info.get('bytes'), int)
                or info['bytes'] <= 0 or not isinstance(info.get('sha256'), str)
                or not re.fullmatch(r'[0-9a-f]{64}', info['sha256'])):
            raise ValueError('Invalid cache file size or SHA256 in manifest')
    fingerprint = hashlib.sha256(''.join(
        f"{name} {meta['files'][name]['sha256']}\n" for name in sorted(required)).encode()).hexdigest()
    if fingerprint != meta.get('fingerprint'):
        raise ValueError('Cache ZIP fingerprint differs from its manifest')
    return required


def verify_cache_zip(path):
    """Check ZIP contents and every uncompressed file hash, not just the ZIP header."""
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        if len(names) != len(set(names)) or set(names) != set(CACHE_ARRAYS + SOURCE_CSVS + ('meta.json',)):
            raise ValueError('Cache ZIP must contain exactly the required files at its root')
        meta = json.loads(z.read('meta.json'))
        for name in _check_cache_manifest(meta):
            info = meta['files'][name]
            digest = hashlib.sha256()
            with z.open(name) as f:
                for block in iter(lambda: f.read(1 << 22), b''):
                    digest.update(block)
            if z.getinfo(name).file_size != info['bytes'] or digest.hexdigest() != info['sha256']:
                raise ValueError(f'Cache ZIP checksum mismatch: {name}')
    return meta


def extract_zip_checked(archive, destination):
    """Extract inside a staging folder, rejecting escaping paths and symlink entries."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (destination / member.filename).resolve()
            if ('\\' in member.filename or not target.is_relative_to(root)
                    or (member.external_attr >> 16) & 0o170000 == 0o120000):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        z.extractall(destination)  # decompression also checks each member's CRC


def publish_data_directory(staged, destination):
    """Keep the previous directory until the validated replacement can be published."""
    staged, destination = Path(staged), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    previous = destination.with_name(f'.{destination.name}.previous-{time.time_ns()}')
    if destination.exists():
        os.replace(destination, previous)
    try:
        os.replace(staged, destination)
    except BaseException:
        if previous.exists():
            try:
                os.replace(previous, destination)
            except OSError:
                # Keep recovery files even if the filesystem also refuses rollback.
                print(f'[data] previous files preserved at {previous}; restore them before retrying')
        raise
    else:
        if previous.is_dir() and not previous.is_symlink():
            shutil.rmtree(previous)
        else:
            previous.unlink(missing_ok=True)


def drive_cache_candidates(folder):
    folder = Path(folder)
    if not folder.is_dir():
        return []
    pattern = re.compile(r'cache(?: \(\d+\))?\.zip$')
    return sorted((p for p in folder.iterdir() if p.is_file() and pattern.fullmatch(p.name)),
                  key=lambda p: (p.stat().st_mtime_ns, p.name), reverse=True)


def fetch_drive_cache(folder, cache_dir, source_dir=None):
    """Try Drive cache copies newest first; never replace a cache with an unverified ZIP."""
    cache_dir = Path(cache_dir)
    cache_dir.parent.mkdir(parents=True, exist_ok=True)
    for archive in drive_cache_candidates(folder):
        try:
            with tempfile.TemporaryDirectory(prefix='.restore-cache-', dir=cache_dir.parent) as tmp:
                local_zip, staged = Path(tmp) / 'cache.zip', Path(tmp) / 'cache'
                shutil.copyfile(archive, local_zip)
                meta = verify_cache_zip(local_zip)
                if source_dir is not None and not _cache_matches_source(meta, source_dir):
                    print(f'[cache] skipping {archive.name}: source CSVs differ')
                    continue
                extract_zip_checked(local_zip, staged)
                verify_cache(staged)
                publish_data_directory(staged, cache_dir)
            print(f'[cache] restored {archive} -> {cache_dir}')
            return cache_dir
        except (OSError, ValueError, KeyError, RuntimeError, zipfile.BadZipFile, zlib.error, EOFError) as exc:
            print(f'[cache] cannot reuse {archive.name}: {type(exc).__name__}; trying another source')
    raise FileNotFoundError('No valid compatible Drive cache was found')


def save_drive_cache(cache_dir, folder):
    """Publish a verified cache.zip once; skip writing an identical valid backup."""
    cache_dir, folder = Path(cache_dir), Path(folder)
    meta = verify_cache(cache_dir)
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder / 'cache.zip'
    if destination.is_file():
        try:
            if verify_cache_zip(destination)['fingerprint'] == meta['fingerprint']:
                print(f'[cache] Drive backup already verified: {destination}')
                return destination
        except (OSError, ValueError, KeyError, RuntimeError, zipfile.BadZipFile, zlib.error, EOFError):
            print('[cache] replacing an invalid Drive backup after verifying its replacement')
    partial = folder / f'.cache-{time.time_ns()}.zip.tmp'
    try:
        with tempfile.TemporaryDirectory(prefix='.export-cache-', dir=cache_dir.parent) as tmp:
            archive = Path(tmp) / 'cache.zip'
            with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as z:
                for name in CACHE_ARRAYS + SOURCE_CSVS + ('meta.json',):
                    z.write(cache_dir / name, arcname=name)
            verify_cache_zip(archive)
            shutil.copyfile(archive, partial)
            verify_cache_zip(partial)  # check the Drive copy before replacing the old backup
            os.replace(partial, destination)
    finally:
        partial.unlink(missing_ok=True)
    print(f'[cache] saved reusable Drive backup: {destination}')
    return destination


def find_raw_data(root, require_images=False):
    """Find CSVs under a wrapper folder, optionally checking every referenced PNG exists."""
    for folder in find_competition_data(root):
        if not all((folder / name).is_file() for name in SOURCE_CSVS):
            continue
        if not require_images:
            return folder
        complete = True
        for split, csv_name in (('train', 'train_labels.csv'), ('test', 'test.csv')):
            frame = pd.read_csv(folder / csv_name)
            if 'filename' not in frame or frame.empty or frame['filename'].isna().any():
                complete = False
                break
            image_root = (folder / f'{split}_images').resolve()
            for filename in frame['filename']:
                image = (image_root / str(filename)).resolve()
                if not image.is_relative_to(image_root) or not image.is_file():
                    complete = False
                    break
            if not complete:
                break
        if complete:
            return folder
    return None


# Kaggle download and login disabled: use manually uploaded data instead.
# def run_kaggle_download():
#     """Use KaggleHub in this notebook process and honor its returned cache/input path."""
#     if not _getenv('KAGGLE_CONFIG_DIR') and (Path('/content') / 'kaggle.json').is_file():
#         os.environ['KAGGLE_CONFIG_DIR'] = '/content'
#     try:
#         kagglehub = importlib.import_module('kagglehub')
#     except ModuleNotFoundError as exc:
#         if exc.name != 'kagglehub':
#             raise RuntimeError('A KaggleHub dependency is missing. Run %pip install -U "kagglehub>=1,<2", '
#                                'restart the kernel, and rerun from the configuration cell.') from None
#         print('[download] installing KaggleHub in this kernel environment')
#         installed = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
#                                     '--disable-pip-version-check', 'kagglehub>=1,<2'],
#                                    capture_output=True, text=True)
#         if installed.returncode:
#             raise RuntimeError('Cannot install KaggleHub. Check internet access and use Python 3.11+; '
#                                'alternatively provide extracted data or a saved cache.')
#         importlib.invalidate_caches()
#         kagglehub = importlib.import_module('kagglehub')
#     print(f'[download] KaggleHub competition_download({KAGGLE_SLUG!r})', flush=True)
#     try:
#         # Do not assume a ZIP filename or pass newer optional API arguments.
#         # This also preserves credentials set by kagglehub.login() in this kernel.
#         downloaded = Path(kagglehub.competition_download(KAGGLE_SLUG))
#     except Exception as exc:
#         response = getattr(exc, 'response', None)
#         status = getattr(response, 'status_code', None)
#         if not isinstance(status, int):
#             status = getattr(exc, 'error_code', None)
#         status_text = f', HTTP {status}' if isinstance(status, int) and 100 <= status <= 599 else ''
#         kind = type(exc).__name__
#         if status == 401 or kind in ('UnauthenticatedError', 'CredentialError'):
#             print('[auth] Kaggle login is needed for this download. Enter your own API token in the login prompt.')
#             try:
#                 kagglehub.login()
#             except Exception as login_error:
#                 raise RuntimeError(
#                     f'Cannot open/complete Kaggle login ({type(login_error).__name__}). '
#                     'For notebook widgets, install ipywidgets and enable widget support in your editor, '
#                     'or use a masked getpass prompt to set KAGGLE_API_TOKEN (see README.md). '
#                     'Then rerun this data cell.') from None
#             # Notebook login displays an asynchronous widget. The kernel must become idle
#             # for its Login button to work; do not block/poll or download again immediately.
#             raise RuntimeError(
#                 f'Data setup paused for Kaggle login ({kind}{status_text}). '
#                 'Complete the login prompt above, then rerun THIS DATA CELL and continue the remaining cells. '
#                 'Do not restart the kernel after login. Existing data and valid caches need no login.') from None
#         if status == 403:
#             advice = ('Your account may need competition access or rule acceptance. '
#                       'If you need to switch credentials, use kagglehub.login() or set KAGGLE_API_TOKEN. '
#                       f'Confirm competition access and accepted rules at https://www.kaggle.com/competitions/{KAGGLE_SLUG}/rules.')
#         elif status == 404:
#             advice = f'Check the competition name {KAGGLE_SLUG!r} and that your account can access it.'
#         elif status == 429:
#             advice = 'Kaggle rate-limited the request. Wait before retrying.'
#         elif kind in ('ConnectionError', 'ConnectTimeout', 'ReadTimeout', 'Timeout'):
#             advice = 'Check the runtime internet connection and retry the data cell.'
#         elif isinstance(exc, OSError):
#             advice = 'Check local free space and write permissions for the KaggleHub cache.'
#         else:
#             advice = ('Check KaggleHub installation and account access; use kagglehub.login() or '
#                       'KAGGLE_API_TOKEN if authentication is required.')
#         # Keep useful diagnostics without echoing request URLs/tokens from exception text.
#         raise RuntimeError(f'KaggleHub download failed ({kind}{status_text}). {advice} '
#                            'Rerun the data cell after correcting the issue.') from None
#     if not downloaded.exists():
#         raise FileNotFoundError(f'KaggleHub returned a path that does not exist: {downloaded}')
#     print(f'[download] KaggleHub returned: {downloaded}')
#     return downloaded
#
#
# def download_competition_data(data_dir):
#     """Normalize a KaggleHub folder/ZIP or a legacy downloaded ZIP into the raw data directory."""
#     data_dir = Path(data_dir)
#     data_dir.parent.mkdir(parents=True, exist_ok=True)
#     legacy_archive = data_dir.parent / 'downloads' / KAGGLE_SLUG / f'{KAGGLE_SLUG}.zip'
#     downloaded = None
#     if legacy_archive.is_file():
#         try:
#             with zipfile.ZipFile(legacy_archive) as z:
#                 if z.testzip() is None:
#                     downloaded = legacy_archive
#         except (OSError, zipfile.BadZipFile, zlib.error, EOFError):
#             pass
#     if downloaded is None:
#         downloaded = run_kaggle_download()
#     else:
#         print(f'[download] reusing completed legacy archive: {downloaded}')
#     if downloaded.is_dir() and (downloaded.resolve().is_relative_to(data_dir.resolve())
#                                 or data_dir.resolve().is_relative_to(downloaded.resolve())):
#         raw = find_raw_data(downloaded, require_images=True)
#         if raw is not None:
#             return raw  # reuse overlapping complete data without moving or modifying the provider cache
#         raise ValueError('KaggleHub cache overlaps DATA_DIR. Choose a separate DATA_DIR for extraction.')
#     with tempfile.TemporaryDirectory(prefix='.extract-data-', dir=data_dir.parent) as tmp:
#         extracted = Path(tmp) / 'extracted'
#         if downloaded.is_dir():
#             # KaggleHub normally already extracted the competition archive. Work on a copy:
#             # its cache (or Kaggle's read-only input mount) must not be moved or unpacked in place.
#             shutil.copytree(downloaded, extracted, copy_function=shutil.copyfile)
#             for directory, _, _ in os.walk(extracted):
#                 Path(directory).chmod(0o700)  # only the working copy must allow nested ZIP extraction
#         elif zipfile.is_zipfile(downloaded):
#             extract_zip_checked(downloaded, extracted)
#         else:
#             raise ValueError(f'Expected a competition folder or ZIP from KaggleHub, received {downloaded}')
#         unpacked = set()
#         for _ in range(4):
#             raw = find_raw_data(extracted, require_images=True)
#             if raw is not None:
#                 publish_data_directory(raw, data_dir)
#                 print(f'[data] ready: {data_dir}')
#                 return data_dir
#             nested = [p for p in extracted.rglob('*.zip') if p not in unpacked]
#             if not nested:
#                 break
#             for path in nested:
#                 target = path.parent
#                 if path.stem in ('train_images', 'test_images'):
#                     with zipfile.ZipFile(path) as z:
#                         if not any(name.startswith(path.stem + '/') for name in z.namelist()):
#                             target = target / path.stem
#                 extract_zip_checked(path, target)
#                 unpacked.add(path)
#     raise ValueError(f'Downloaded files at {downloaded} lack complete train_images/, test_images/ and '
#                      'the three CSVs. Inspect that folder or set DATA_DIR to complete extracted data. '
#                      'If the KaggleHub cache is incomplete, refresh it with '
#                      'kagglehub.competition_download(..., force_download=True).')
#
#

def prepare_uploaded_data(uploaded, data_dir):
    """Read a user-supplied folder/ZIP; unpack only into local staging, without network access."""
    uploaded, data_dir = Path(uploaded), Path(data_dir)
    if not uploaded.exists():
        raise FileNotFoundError(
            f'No complete data or valid cache. Upload the competition ZIP or extracted files to '
            f'{uploaded}. On Colab the default is MyDrive/DL_Project1/data/. '
            'Kaggle downloads and login are disabled.')
    if (uploaded.resolve().is_relative_to(data_dir.resolve())
            or data_dir.resolve().is_relative_to(uploaded.resolve())):
        raise ValueError('Uploaded source and local extraction directory must be separate.')
    data_dir.parent.mkdir(parents=True, exist_ok=True)
    print(f'[data] reading manually uploaded files: {uploaded}')
    with tempfile.TemporaryDirectory(prefix='.extract-data-', dir=data_dir.parent) as tmp:
        extracted = Path(tmp) / 'extracted'
        if uploaded.is_dir():
            shutil.copytree(uploaded, extracted, copy_function=shutil.copyfile)
            for directory, _, _ in os.walk(extracted):
                Path(directory).chmod(0o700)
        elif zipfile.is_zipfile(uploaded):
            extract_zip_checked(uploaded, extracted)
        else:
            raise ValueError(f'Expected an uploaded competition folder or ZIP, received {uploaded}')
        unpacked = set()
        for _ in range(4):
            raw = find_raw_data(extracted, require_images=True)
            if raw is not None:
                publish_data_directory(raw, data_dir)
                print(f'[data] ready: {data_dir}')
                return data_dir
            nested = [p for p in extracted.rglob('*.zip') if p not in unpacked]
            if not nested:
                break
            for path in nested:
                target = path.parent
                if path.stem in ('train_images', 'test_images'):
                    with zipfile.ZipFile(path) as z:
                        if not any(name.startswith(path.stem + '/') for name in z.namelist()):
                            target = target / path.stem
                extract_zip_checked(path, target)
                unpacked.add(path)
    raise ValueError(
        f'Uploaded files at {uploaded} lack complete train_images/, test_images/, '
        'train_labels.csv, test.csv and sample_submission.csv. Upload all competition files; '
        'ZIP and extracted-folder layouts are supported.')


def prepare_data(paths, auto_download=False, save_to_drive=True):
    """Valid local cache -> compatible Drive cache -> manually uploaded folder/ZIP."""
    if auto_download:
        raise ValueError("Kaggle download/login is disabled. Keep AUTO_DOWNLOAD=False and upload data manually.")
    cache = paths.cache_dir
    persist = paths.env == 'colab' and save_to_drive
    if persist and not drive_mounted():
        raise RuntimeError('Mount Google Drive before data setup to save cache.zip, or explicitly set '
                           'SAVE_CACHE_TO_DRIVE=False for temporary storage only.')
    source = find_raw_data(paths.data_dir) or paths.data_dir
    meta = None
    if (cache / 'meta.json').is_file():
        try:
            candidate = verify_cache(cache)
            if _cache_matches_source(candidate, source):
                meta = candidate
                print(f'[cache] reusing verified local cache: {cache}')
            else:
                print('[cache] source CSVs changed; rebuilding from the requested data')
        except (OSError, ValueError, KeyError, RuntimeError, EOFError) as exc:
            print(f'[cache] local cache is incomplete or invalid ({type(exc).__name__})')
    if meta is None and paths.env == 'colab' and drive_mounted():
        try:
            fetch_drive_cache(drive_dir(), cache, source)
            meta = verify_cache(cache)
        except FileNotFoundError:
            pass
    if meta is None:
        if paths.data_dir.resolve() == cache.resolve():
            raise ValueError('Use separate DATA_DIR and CACHE_DIR when building from uploaded data')
        raw = find_raw_data(paths.data_dir, require_images=True)
        if raw is None:
            raw = prepare_uploaded_data(paths.data_dir, cache.parent / 'data_extracted')
        paths.data_dir = raw
        cache.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='.build-cache-', dir=cache.parent) as tmp:
            staged = Path(tmp) / 'cache'
            build_cache(raw, staged)
            meta = verify_cache(staged)
            publish_data_directory(staged, cache)
    else:
        paths.data_dir = source
    if persist:
        save_drive_cache(cache, drive_dir())
    return cache, meta


In [ ]:
CACHE, META = prepare_data(PATHS, auto_download=AUTO_DOWNLOAD, save_to_drive=SAVE_CACHE_TO_DRIVE)
# Keep later training/evaluation cells on the resolved source folder (including wrapper directories).
DATA_DIR = str(PATHS.data_dir)
CACHE_DIR = str(CACHE)
print(f"[cache] {CACHE}: format {META['format']} | {META['n_train']:,} train / {META['n_test']:,} test images "
      f"{tuple(META['shape'])} | fingerprint {META['fingerprint'][:12]} | sha256 verified")
counts = np.bincount(np.load(CACHE / "train_labels.npy"), minlength=NUM_CLASSES)
print("Training images per class: " + ", ".join(f"{c} {n:,}" for c, n in zip(CLASSES, counts)))

In [ ]:
# Verbatim copies of training-notebook helpers (split, inference, checkpoint identity). Do not edit:
# tests/test_notebooks.py compares every definition below with all six training notebooks.
# model_fingerprint, validate_checkpoint and load_model use family names (MODEL, build_model, ...);
# call them through family_namespace(model) from section 5, never directly.

def _per_class(n, what):
    if not isinstance(n, (int, np.integer)) or isinstance(n, (bool, np.bool_)) or n < 0:
        raise ValueError(f"{what} must be a nonnegative integer, got {n!r}")
    if n % NUM_CLASSES:
        raise ValueError(f"{what} must be a multiple of {NUM_CLASSES} for a class-balanced split, got {n}")
    return n // NUM_CLASSES


def stratified_split(labels, val_size, seed=0):
    """Hold out `val_size` images, the same number from every class. Returns sorted (train_idx, val_idx)."""
    labels = np.asarray(labels)
    if labels.ndim != 1 or not np.issubdtype(labels.dtype, np.integer) or not len(labels):
        raise ValueError("labels must be a nonempty one-dimensional integer array")
    if np.any((labels < 0) | (labels >= NUM_CLASSES)):
        raise ValueError("labels must be in 0..9")
    k = _per_class(val_size, "val_size")
    if val_size >= len(labels):
        raise ValueError("Validation size must leave at least one training image")
    if np.any(np.bincount(labels, minlength=NUM_CLASSES) < k):
        raise ValueError("Not enough examples per class for the requested validation split")
    all_idx = np.arange(len(labels))
    if val_size == 0:
        return all_idx, np.array([], dtype=np.int64)
    k = _per_class(val_size, "val_size")
    rng = np.random.default_rng(seed)
    val = np.concatenate([rng.permutation(np.flatnonzero(labels == c))[:k] for c in range(NUM_CLASSES)])
    val = np.sort(val)
    return np.setdiff1d(all_idx, val), val


def to_float_nchw(x_u8, mean, std, channels_last=False):
    """uint8 NHWC on device -> normalized float NCHW (channels_last strides are free from NHWC)."""
    x = x_u8.permute(0, 3, 1, 2).float().div_(255.0)
    x = (x - mean) / std
    return x.contiguous(memory_format=torch.channels_last if channels_last else torch.contiguous_format)


def replace_file(src, dst):
    """Publish a completed file by rename; never overwrite a good checkpoint by partial copy."""
    try:
        os.replace(src, dst)
    except OSError as exc:
        raise OSError(f"Could not atomically publish {dst}; the previous file is preserved. "
                      "Use a filesystem that supports same-directory rename.") from exc


def write_bytes_atomic(data, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(data)
    replace_file(tmp, path)


def load_checkpoint(path):
    state = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(state, dict) or "args" not in state:
        return state
    original = state["args"].get("run_name")
    canonical = canonical_run_id(original)
    if original != canonical:
        state.setdefault("original_identity", {"run_id": original,
                         "model_version": state["args"].get("model_version")})
        state["args"] = dict(state["args"], run_name=canonical,
                             model_version=int(re.search(r"-ver-(\d+)-", canonical).group(1)),
                             task_id=run_task_id(canonical))
    return state


@torch.inference_mode()
def predict_logits(model, images_u8, mean, std, device, batch_size=1000, flip=False, channels_last=False):
    """Logits for uint8 NHWC images (numpy or tensor); returns a CPU float tensor.

    With flip=True, returns the logits of horizontally flipped images instead.
    """
    model.eval()
    mean_t = torch.tensor(mean, device=device).view(1, 3, 1, 1)
    std_t = torch.tensor(std, device=device).view(1, 3, 1, 1)
    out = []
    for i in range(0, len(images_u8), batch_size):
        xb = torch.as_tensor(np.asarray(images_u8[i:i + batch_size])).to(device)
        x = to_float_nchw(xb, mean_t, std_t, channels_last)
        if flip:
            x = x.flip(3)
        out.append(model(x).float().cpu())
    return torch.cat(out)


CHECKPOINT_FORMAT = 3


def write_json_atomic(value, path):
    write_bytes_atomic((json.dumps(value, indent=2) + "\n").encode(), path)


def model_fingerprint(model):
    options = getattr(model, "_project_architecture", ARCH_DEFAULTS)
    definition = getattr(model, "_project_description", MODEL_DESCRIPTION) + "\n" + repr(model)
    # Preserve the exact legacy baseline hash. New variants also bind options and implementation version.
    if options != ARCH_DEFAULTS:
        definition += "\n" + json.dumps(options, sort_keys=True)
    return hashlib.sha256(definition.encode()).hexdigest()


def validate_checkpoint(ckpt, fingerprint, expected_model_hash=None, run_id=None):
    if ckpt.get("format") != CHECKPOINT_FORMAT or "cache_fingerprint" not in ckpt:
        raise ValueError("Legacy checkpoint lacks verified dataset identity; start a fresh run with this notebook")
    if ckpt["cache_fingerprint"] != fingerprint:
        raise ValueError("Checkpoint dataset fingerprint differs from the current cache; refusing resume/evaluation")
    if ckpt["args"]["model"] != MODEL:
        raise ValueError("Checkpoint belongs to another model")
    saved_options = saved_architecture(ckpt["args"])
    if "architecture" in ckpt and ckpt["architecture"] != architecture_details(saved_options):
        raise ValueError("Checkpoint resolved architecture differs from its saved options")
    if expected_model_hash is not None and ckpt.get("model_fingerprint") != expected_model_hash:
        raise ValueError("Checkpoint architecture differs from this notebook")
    if run_id is not None and canonical_run_id(ckpt["args"]["run_name"]) != canonical_run_id(run_id):
        raise ValueError("Checkpoint belongs to another run ID")


def load_model(ckpt, weights, device):
    options = saved_architecture(ckpt["args"])
    model, n_params = build_model(architecture=options)
    if "architecture" in ckpt and ckpt["architecture"] != architecture_details(options):
        raise ValueError("Checkpoint resolved architecture differs from its saved options")
    if "n_params" in ckpt and ckpt["n_params"] != n_params:
        raise ValueError("Checkpoint parameter count differs from its saved architecture")
    if ckpt.get("model_fingerprint") != model_fingerprint(model) or ckpt["args"]["model"] != MODEL:
        raise ValueError("Checkpoint architecture differs from this notebook")
    model.load_state_dict(ckpt["ema"]["module"] if weights == "ema" else ckpt["model"])
    return model.to(device).eval(), n_params


def classification_metrics(probs, labels):
    labels = np.asarray(labels)
    probs = np.asarray(probs)
    pred = probs.argmax(1)
    cm = np.bincount(labels * NUM_CLASSES + pred, minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)
    support = cm.sum(1)
    recall = np.divide(cm.diagonal(), support, out=np.zeros(NUM_CLASSES, dtype=float), where=support > 0)
    return {"accuracy_pct": float((pred == labels).mean()*100), "correct": int((pred == labels).sum()),
            "n": len(labels), "cross_entropy": float(-np.log(probs[np.arange(len(labels)), labels].clip(1e-12)).mean()),
            "per_class_recall": {name: float(value) if count else None
                                 for name, value, count in zip(CLASSES, recall, support)},
            "confusion_matrix": cm.tolist()}

In [ ]:
# ---- the training notebooks' exact 5k validation split
COMPETITION_CACHE_FINGERPRINT = "42749a486a13184f0f96dab13293950f0120c357ddf1e6b1ef6a3bd3a9ce5fde"
# SHA-256 of the sorted int64 validation indices and their labels. Both equal every validation export
# saved by the A/B/C/E/F/G runs (result/evaluations/validation/<RUN_ID>/*-validation.npz).
EXPECTED_VAL_INDICES_SHA256 = "7957d060fcc34aefa5a43c70e31433927db55526494f67a9f7da553059be6767"
EXPECTED_VAL_LABELS_SHA256 = "aba30157995bda52486927048b705427db4a51e03e0bcb4844d462bb15f20846"


def int64_sha256(values):
    """SHA-256 of little-endian int64 bytes (indices and labels)."""
    return hashlib.sha256(np.ascontiguousarray(values, dtype="<i8").tobytes()).hexdigest()


if Q08_COMPETITION_CHECKS:
    if META["fingerprint"] != COMPETITION_CACHE_FINGERPRINT:
        raise RuntimeError(f"Cache fingerprint {META['fingerprint'][:12]} is not the competition dataset "
                           f"{COMPETITION_CACHE_FINGERPRINT[:12]}; Q08 must use the training notebooks' data")
    if (VAL_SIZE, SPLIT_SEED) != (5000, 0):
        raise ValueError("Q08 uses the training notebooks' split: keep VAL_SIZE=5000 and SPLIT_SEED=0")
_train_images, _train_labels, _train_ids = load_train(DATA_DIR, CACHE_DIR, mmap=True)
TRAIN_IDX, VAL_IDX = stratified_split(_train_labels, VAL_SIZE, SPLIT_SEED)
if not len(VAL_IDX):
    raise ValueError("Q08 needs a validation split (VAL_SIZE > 0)")
VAL_LABELS_ALL = np.ascontiguousarray(_train_labels[VAL_IDX], dtype=np.int64)
if Q08_COMPETITION_CHECKS:
    if int64_sha256(VAL_IDX) != EXPECTED_VAL_INDICES_SHA256 or int64_sha256(VAL_LABELS_ALL) != EXPECTED_VAL_LABELS_SHA256:
        raise RuntimeError("Validation indices or labels differ from the training notebooks' 5k split")
if type(Q08_SMOKE_N) is not int or Q08_SMOKE_N < 0:
    raise ValueError("Q08_SMOKE_N must be a nonnegative integer")
Q08_N = min(Q08_SMOKE_N, len(VAL_IDX)) if Q08_SMOKE_N else len(VAL_IDX)
Q08_SMOKE = Q08_N != len(VAL_IDX)
VAL_X = np.ascontiguousarray(_train_images[VAL_IDX[:Q08_N]])  # raw uint8 NHWC, before mean/std normalization
VAL_Y = VAL_LABELS_ALL[:Q08_N]
del _train_images, _train_labels, _train_ids
print(f"[split] {len(TRAIN_IDX):,} train / {len(VAL_IDX):,} validation (seed {SPLIT_SEED}); "
      f"indices sha256 {int64_sha256(VAL_IDX)[:12]} | labels sha256 {int64_sha256(VAL_LABELS_ALL)[:12]} | "
      f"cache {META['fingerprint'][:12]}" + (" | competition split verified" if Q08_COMPETITION_CHECKS else ""))
print("Validation images per class: " + ", ".join(
    f"{c} {n}" for c, n in zip(CLASSES, np.bincount(VAL_LABELS_ALL, minlength=NUM_CLASSES))))
if Q08_SMOKE:
    print(f"Warning: Q08_SMOKE_N={Q08_SMOKE_N}: using the first {Q08_N} validation images; results are a smoke "
          "test, not comparable, and the reference manifest is neither written nor checked")

## 4. Shifted validation set (Part 1)

Thirteen corruptions × severities 1–3, applied to the raw uint8 32×32 validation images. Values marked CIFAR-10-C
are copied from `make_cifar_c.py` in [hendrycks/robustness](https://github.com/hendrycks/robustness/blob/master/ImageNet-C/create_c/make_cifar_c.py)
(file last changed in commit `ca36008`, SHA-256 `f865ae91…` of the fetched file); all others are defined here by
the stated symmetry, **before** any model was evaluated on the set.

| Family | Corruption | Severity 1 / 2 / 3 | Source |
|---|---|---|---|
| photometric | `brightness_up` | HSV value V + 0.05 / 0.10 / 0.15 | CIFAR-10-C brightness 1–3 |
| photometric | `brightness_down` | V − 0.05 / 0.10 / 0.15 | defined: mirror (same magnitudes, opposite sign) |
| photometric | `contrast_down` | (x − μ)·0.75 / 0.5 / 0.4 + μ, μ per image and channel | CIFAR-10-C contrast 1–3 |
| photometric | `contrast_up` | factor 1/0.75 / 1/0.5 / 1/0.4 = 1.33 / 2.0 / 2.5 | defined: reciprocal, the exact inverse before clipping |
| photometric | `saturate_down` | HSV saturation S × 0.667 / 0.3 / 0.1 | s1 defined (1/1.5, inverse of CIFAR-10-C's first increase); s2–s3 = CIFAR-10-C saturate levels 1–2 |
| photometric | `saturate_up` | S × 1.5 / (2.0, +0.1) / (2.5, +0.2) | CIFAR-10-C saturate levels 3–5 (its only increases) |
| photometric | `gamma_bright` | x^0.8625 / x^0.7370 / x^0.6215 | defined: 0.5^g = 0.5 + c for the brightness c = 0.05 / 0.10 / 0.15 |
| photometric | `gamma_dark` | x^1.1520 / x^1.3219 / x^1.5146 | defined: 0.5^g = 0.5 − c, same c |
| sharpness | `gaussian_blur` | σ = 0.4 / 0.6 / 0.7 px | CIFAR-10-C gaussian_blur 1–3 |
| sharpness | `sharpen` | unsharp mask 2x − G_σ(x), σ = 0.4 / 0.6 / 0.7 | defined: same σ; to first order the inverse of that blur |
| noise | `gaussian_noise` | additive N(0, σ²) per pixel and channel, σ = 0.04 / 0.06 / 0.08 of the [0, 1] range (≈10 / 15 / 20 levels) | CIFAR-10-C gaussian_noise 1–3 |
| compression | `jpeg` | Pillow JPEG quality 80 / 65 / 58 | CIFAR-10-C jpeg_compression 1–3 |
| compression | `pixelate` | Pillow BOX to 30 / 28 / 27 px and back to 32 | CIFAR-10-C pixelate 1–3 (int(32·0.95/0.90/0.85)) |

**Why these defined values.** Brightness and darkening use the same additive V step. Contrast up uses 1/c, which
undoes the CIFAR-10-C contrast reduction exactly (before clipping). CIFAR-10-C's `saturate` orders its levels by
difficulty, not direction: levels 1–2 desaturate (×0.3, ×0.1) and 3–5 saturate (×1.5, ×2+0.1, ×2.5+0.2). Q08 splits
the two directions, so `saturate_up` uses levels 3–5 and `saturate_down` adds ×1/1.5 as its light level. Gamma is a
nonlinear tone curve that keeps black and white fixed; its strengths move mid-grey by exactly the brightness steps.
For a small σ, a Gaussian blur adds about +σ²/2·∇²x and the unsharp mask with amount 1 subtracts it, so `sharpen` is
the mirror of `gaussian_blur`. These magnitudes come only from the CIFAR-10-C values and the symmetries above.

**Faithfulness and deviations.** Brightness and saturation are implemented in closed form: with hue and the other
HSV component fixed, RGB scales with V (brightness) or moves toward V (saturation). This equals CIFAR-10-C's skimage
`rgb2hsv → hsv2rgb` round trip in exact arithmetic, including skimage's convention that grey pixels have S = 0 and
hue 0. Gaussian blur reproduces skimage `gaussian` (scipy kernel, `nearest` edges, truncate 4, channels not mixed).
Every operation works in float64 on x/255, clips to [0, 1] and **rounds** to uint8; CIFAR-10-C instead truncates with
`np.uint8`, so its outputs can be at most one level lower. The noise uses an independent NumPy PCG64 stream per
corruption × severity (seeded from SHA-256 of `"Q08|<name>|<severity>"`), not CIFAR-10-C's global RNG, so values
differ while the distribution is the same.

**Determinism and sharing.** NumPy-only sets must reproduce bit for bit; a mismatch stops the notebook. JPEG and
pixelate use Pillow/libjpeg exactly as CIFAR-10-C does, and their bytes can differ between library builds (for
example libjpeg-turbo on Colab versus IJG libjpeg in some local conda builds), so a mismatch there prints a clear
warning with both library versions (`Q08_REQUIRE_EXACT=True` makes it an error). The shared `.npz` removes that
dependence: its stored bytes are the set.

In [ ]:
# ---- Q08 corruption set: fixed a priori, version-locked (bump Q08_SET_VERSION for ANY change below)
Q08_SET_VERSION = 1
Q08_SET_NAME = f"Q08_shiftval-v{Q08_SET_VERSION}"
Q08_FAMILIES = ("photometric", "sharpness", "noise", "compression")
CIFAR10C_REFERENCE = {
    "paper": "Hendrycks & Dietterich (2019), Benchmarking Neural Network Robustness to Common Corruptions and Perturbations",
    "repository": "https://github.com/hendrycks/robustness",
    "file": "ImageNet-C/create_c/make_cifar_c.py",
    "file_last_commit": "ca360085d57889288c4143fddd8a707bbcae01bd",
    "file_sha256": "f865ae91dffb2b4834292ce05c876bf3d77f36417f70075a2b5259348b0c83a5",
}
# params[k] is severity k+1. "source" says which values come from CIFAR-10-C (make_cifar_c.py) and which are
# defined here by a stated symmetry. None is fitted to hidden-test statistics or tuned on model accuracy.
Q08_CORRUPTIONS = {
    "brightness_up":   {"family": "photometric", "op": "brightness", "params": [0.05, 0.10, 0.15],
                        "source": "CIFAR-10-C brightness severities 1-3: HSV value V + c, clipped to [0, 1]"},
    "brightness_down": {"family": "photometric", "op": "brightness", "params": [-0.05, -0.10, -0.15],
                        "source": "defined: mirror of CIFAR-10-C brightness (HSV value V - c, same magnitudes)"},
    "contrast_down":   {"family": "photometric", "op": "contrast", "params": [0.75, 0.5, 0.4],
                        "source": "CIFAR-10-C contrast severities 1-3: (x - per-image channel mean) * c + mean"},
    "contrast_up":     {"family": "photometric", "op": "contrast", "params": [1 / 0.75, 1 / 0.5, 1 / 0.4],
                        "source": "defined: reciprocal factors 1/c of CIFAR-10-C contrast (exact inverse before clipping)"},
    "saturate_down":   {"family": "photometric", "op": "saturate", "params": [[1 / 1.5, 0.0], [0.3, 0.0], [0.1, 0.0]],
                        "source": "severity 1 defined as 1/1.5 (inverse of CIFAR-10-C saturate level 3); severities 2-3 = "
                                  "CIFAR-10-C saturate levels 1-2 (HSV S * a + b)"},
    "saturate_up":     {"family": "photometric", "op": "saturate", "params": [[1.5, 0.0], [2.0, 0.1], [2.5, 0.2]],
                        "source": "CIFAR-10-C saturate levels 3-5, its only saturation increases (levels 1-2 desaturate)"},
    "gamma_bright":    {"family": "photometric", "op": "gamma", "params": [0.8625, 0.7370, 0.6215],
                        "source": "defined: x ** g with 0.5 ** g = 0.5 + c for the CIFAR-10-C brightness c = 0.05/0.10/0.15"},
    "gamma_dark":      {"family": "photometric", "op": "gamma", "params": [1.1520, 1.3219, 1.5146],
                        "source": "defined: x ** g with 0.5 ** g = 0.5 - c for the CIFAR-10-C brightness c = 0.05/0.10/0.15"},
    "gaussian_blur":   {"family": "sharpness", "op": "gaussian_blur", "params": [0.4, 0.6, 0.7],
                        "source": "CIFAR-10-C gaussian_blur severities 1-3: sigma (skimage gaussian, nearest edges, truncate 4)"},
    "sharpen":         {"family": "sharpness", "op": "unsharp_mask", "params": [0.4, 0.6, 0.7],
                        "source": "defined: unsharp mask 2x - G_sigma(x) with the CIFAR-10-C blur sigma; to first order the "
                                  "inverse of that blur"},
    "gaussian_noise":  {"family": "noise", "op": "gaussian_noise", "params": [0.04, 0.06, 0.08],
                        "source": "CIFAR-10-C gaussian_noise severities 1-3: additive N(0, c^2) in [0, 1] units"},
    "jpeg":            {"family": "compression", "op": "jpeg", "params": [80, 65, 58],
                        "source": "CIFAR-10-C jpeg_compression severities 1-3: Pillow JPEG quality"},
    "pixelate":        {"family": "compression", "op": "pixelate", "params": [30, 28, 27],
                        "source": "CIFAR-10-C pixelate severities 1-3: int(32 * c) for c = 0.95/0.90/0.85, Pillow BOX down then up"},
}
Q08_SEVERITIES = (1, 2, 3)
# Severity 0 applies each operation's identity parameter (JPEG has none: severity 0 is the clean copy).
Q08_IDENTITY_PARAMS = {"brightness": 0.0, "contrast": 1.0, "saturate": [1.0, 0.0], "gamma": 1.0,
                       "gaussian_blur": 0.0, "unsharp_mask": 0.0, "gaussian_noise": 0.0, "jpeg": None, "pixelate": 32}
Q08_LIBRARY_DEPENDENT_OPS = ("jpeg", "pixelate")  # Pillow/libjpeg code paths; all other operations are NumPy only
Q08_QUANTIZATION = ("float64 arithmetic on x/255; clip to [0, 1]; round half to even (numpy.rint) to uint8. "
                    "CIFAR-10-C truncates with np.uint8 instead (at most one level lower).")
# NumPy-only operations use correctly rounded elementwise arithmetic, exact integer sums, and scalar math/pow
# for the few kernel/curve values, so their bytes do not depend on summation order or SIMD code paths.
Q08_CHUNK = 1000  # images per generation chunk (bounds memory; identical results for any chunk size)
_PIL_BOX = getattr(Image, "Resampling", Image).BOX


def q08_set_key(name, severity):
    return f"{name}__s{severity}"


def q08_set_keys():
    return [q08_set_key(name, s) for name in Q08_CORRUPTIONS for s in Q08_SEVERITIES]


def q08_key_info(key):
    name, severity = key.rsplit("__s", 1)
    spec = Q08_CORRUPTIONS[name]
    return name, int(severity), spec


def q08_library_dependent(key):
    """True for sets made by Pillow/libjpeg code, whose bytes may differ between library builds."""
    name = key.rsplit("__s", 1)[0]
    return name in Q08_CORRUPTIONS and Q08_CORRUPTIONS[name]["op"] in Q08_LIBRARY_DEPENDENT_OPS


def _unit(x_u8):
    return np.asarray(x_u8, dtype=np.float64) / 255.0


def _quantize(x):
    return np.rint(np.clip(x, 0.0, 1.0) * 255.0).astype(np.uint8)


def op_brightness(x_u8, c):
    """HSV value V -> clip(V + c); hue and saturation fixed (CIFAR-10-C via skimage rgb2hsv/hsv2rgb).

    With H and S fixed, RGB scales by V'/V. Black (V = 0) has S = 0 there, so it becomes grey V'.
    This closed form equals the HSV round trip in exact arithmetic and avoids hue round-off.
    """
    x = _unit(x_u8)
    v = x.max(axis=-1, keepdims=True)
    v_new = np.clip(v + c, 0.0, 1.0)
    scale = np.divide(v_new, v, out=np.zeros_like(v), where=v > 0)
    return _quantize(np.where(v > 0, x * scale, v_new))


def op_contrast(x_u8, c):
    """(x - mean) * c + mean with the per-image, per-channel mean (CIFAR-10-C contrast).

    The mean is an exact integer sum divided once, so it does not depend on NumPy's summation order.
    """
    x_u8 = np.asarray(x_u8)
    x = _unit(x_u8)
    means = x_u8.sum(axis=(1, 2), keepdims=True, dtype=np.int64) / (255.0 * x_u8.shape[1] * x_u8.shape[2])
    return _quantize((x - means) * c + means)


def op_saturate(x_u8, params):
    """HSV saturation S -> clip(S * a + b); hue and value fixed (CIFAR-10-C saturate, closed form).

    A coloured channel is V - V*S*w(H), so it becomes V - (V - x) * S'/S. Grey pixels have S = 0 and hue 0
    (red) in skimage, so S' = b > 0 gives (V, V(1 - b), V(1 - b)); b = 0 leaves them unchanged.
    """
    a, b = params
    x = _unit(x_u8)
    v = x.max(axis=-1, keepdims=True)
    delta = v - x.min(axis=-1, keepdims=True)
    s = np.divide(delta, v, out=np.zeros_like(v), where=delta > 0)
    s_new = np.clip(s * a + b, 0.0, 1.0)
    ratio = np.divide(s_new, s, out=np.zeros_like(s), where=delta > 0)
    coloured = v - (v - x) * ratio
    grey = np.concatenate([v, v * (1.0 - s_new), v * (1.0 - s_new)], axis=-1)
    return _quantize(np.where(delta > 0, coloured, grey))


def op_gamma(x_u8, g):
    """Per-channel power curve x ** g on [0, 1] (g < 1 brightens mid-tones, g > 1 darkens; 0 and 1 fixed)."""
    lut = _quantize(np.array([(k / 255.0) ** g for k in range(256)], dtype=np.float64))
    return lut[np.asarray(x_u8)]


def _gaussian_kernel(sigma, truncate=4.0):
    """scipy.ndimage 1D Gaussian kernel (as used by skimage.filters.gaussian); sigma 0 is the identity."""
    radius = int(truncate * float(sigma) + 0.5) if sigma > 0 else 0
    if radius == 0:
        return np.ones(1)
    # Scalar math.exp: the few weights do not depend on NumPy's vectorized exp implementation.
    weights = np.array([math.exp(-0.5 / (sigma * sigma) * t * t) for t in range(-radius, radius + 1)])
    return weights / weights.sum()


def _blur_hw(x, sigma):
    """Separable Gaussian over height then width of float NHWC images; edges replicate ('nearest')."""
    w = _gaussian_kernel(sigma)
    r = len(w) // 2
    if r == 0:
        return x.copy()
    h, wd = x.shape[1:3]
    p = np.pad(x, ((0, 0), (r, r), (0, 0), (0, 0)), mode="edge")
    out = sum(w[i] * p[:, i:i + h] for i in range(len(w)))
    p = np.pad(out, ((0, 0), (0, 0), (r, r), (0, 0)), mode="edge")
    return sum(w[i] * p[:, :, i:i + wd] for i in range(len(w)))


def op_gaussian_blur(x_u8, sigma):
    """CIFAR-10-C gaussian_blur: skimage gaussian(sigma, multichannel) = per-channel spatial blur."""
    return _quantize(_blur_hw(_unit(x_u8), sigma))


def op_unsharp_mask(x_u8, sigma, amount=1.0):
    """x + amount * (x - G_sigma(x)); amount 1 subtracts the blur's first-order (sigma^2/2) Laplacian term."""
    x = _unit(x_u8)
    return _quantize(x + amount * (x - _blur_hw(x, sigma)))


def op_gaussian_noise(x_u8, sigma, rng):
    """CIFAR-10-C gaussian_noise: x + N(0, sigma^2) per pixel and channel, from this set's own RNG stream."""
    x = _unit(x_u8)
    noise = rng.standard_normal(size=x.shape, dtype=np.float64)  # drawn even for sigma 0: same stream position
    return _quantize(x + sigma * noise)


def op_jpeg(x_u8, quality):
    """CIFAR-10-C jpeg_compression: Pillow JPEG encode at `quality` (default 4:2:0 subsampling) and decode."""
    if quality is None:
        return np.array(x_u8, dtype=np.uint8, copy=True)
    out = np.empty_like(x_u8)
    for i, img in enumerate(x_u8):
        buffer = io.BytesIO()
        Image.fromarray(img).save(buffer, format="JPEG", quality=int(quality))
        buffer.seek(0)
        with Image.open(buffer) as decoded:
            out[i] = np.asarray(decoded.convert("RGB"), dtype=np.uint8)
    return out


def op_pixelate(x_u8, size):
    """CIFAR-10-C pixelate: Pillow BOX resize to size x size, then BOX back to the original size."""
    h, w = x_u8.shape[1:3]
    out = np.empty_like(x_u8)
    for i, img in enumerate(x_u8):
        small = Image.fromarray(img).resize((int(size), int(size)), _PIL_BOX)
        out[i] = np.asarray(small.resize((w, h), _PIL_BOX), dtype=np.uint8)
    return out


Q08_OPS = {"brightness": op_brightness, "contrast": op_contrast, "saturate": op_saturate, "gamma": op_gamma,
           "gaussian_blur": op_gaussian_blur, "unsharp_mask": op_unsharp_mask, "gaussian_noise": op_gaussian_noise,
           "jpeg": op_jpeg, "pixelate": op_pixelate}


def q08_rng(name, severity):
    """Independent stream per corruption x severity: PCG64 seeded by SHA-256("Q08|<name>|<severity>")."""
    digest = hashlib.sha256(f"Q08|{name}|{severity}".encode()).digest()
    return np.random.Generator(np.random.PCG64(np.random.SeedSequence(int.from_bytes(digest[:16], "little"))))


def q08_corrupt(x_u8, name, severity, rng=None, chunk=Q08_CHUNK):
    """Corrupt uint8 NHWC images; images are processed in order, so any chunk size gives the same bytes."""
    x_u8 = np.asarray(x_u8)
    if x_u8.dtype != np.uint8 or x_u8.ndim != 4 or x_u8.shape[-1] != 3:
        raise ValueError("Q08 corruptions take uint8 NHWC RGB images")
    spec = Q08_CORRUPTIONS[name]
    if severity not in (0,) + Q08_SEVERITIES:
        raise ValueError(f"severity must be 0 (identity) or one of {Q08_SEVERITIES}")
    param = Q08_IDENTITY_PARAMS[spec["op"]] if severity == 0 else spec["params"][severity - 1]
    op = Q08_OPS[spec["op"]]
    rng = rng if rng is not None else q08_rng(name, severity)
    out = np.empty_like(x_u8)
    for start in range(0, len(x_u8), max(1, int(chunk))):
        part = x_u8[start:start + chunk]
        out[start:start + chunk] = op(part, param, rng) if spec["op"] == "gaussian_noise" else op(part, param)
    return out


def q08_spec_record():
    """JSON-ready description of the locked set (also hashed into the manifest)."""
    return {"set_name": Q08_SET_NAME, "set_version": Q08_SET_VERSION, "families": list(Q08_FAMILIES),
            "severities": list(Q08_SEVERITIES), "corruptions": Q08_CORRUPTIONS, "quantization": Q08_QUANTIZATION,
            "seeding": "numpy PCG64(SeedSequence(int.from_bytes(sha256('Q08|<name>|<severity>')[:16], 'little')))",
            "cifar10c_reference": CIFAR10C_REFERENCE}


def q08_generate(clean, verbose=True):
    """All corruption x severity copies of `clean` (uint8 NHWC) stacked as (n_sets, N, 32, 32, 3) uint8."""
    keys = q08_set_keys()
    stack = np.empty((len(keys),) + clean.shape, dtype=np.uint8)
    t0 = time.perf_counter()
    for i, key in enumerate(keys):
        name, severity, _ = q08_key_info(key)
        stack[i] = q08_corrupt(clean, name, severity)
    if verbose:
        print(f"[q08] generated {len(keys)} corruption x severity sets of {len(clean):,} images "
              f"in {time.perf_counter() - t0:.1f}s ({stack.nbytes / 2**20:.0f} MiB uint8)")
    return keys, stack

In [ ]:
# ---- fingerprints, reference manifest and the optional shared .npz
Q08_FIXED_KEYS = ("clean", "labels", "val_indices")


def q08_array_sha256(a):
    """SHA-256 over dtype, shape and C-order bytes, so equal hashes mean bit-identical arrays."""
    a = np.ascontiguousarray(a)
    digest = hashlib.sha256(f"{a.dtype.str}|{'x'.join(str(n) for n in a.shape)}|".encode())
    digest.update(a.tobytes())
    return digest.hexdigest()


def q08_fingerprints(clean, labels, indices, keys, stack):
    hashes = {"clean": q08_array_sha256(clean),
              "labels": q08_array_sha256(np.asarray(labels, dtype="<i8")),
              "val_indices": q08_array_sha256(np.asarray(indices, dtype="<i8"))}
    hashes.update({key: q08_array_sha256(stack[i]) for i, key in enumerate(keys)})
    return hashes


def q08_set_sha256(hashes, version=Q08_SET_VERSION):
    """Whole-set fingerprint: the set version plus every array hash, sorted by name."""
    lines = [f"set_version {version}"] + [f"{key} {hashes[key]}" for key in sorted(hashes)]
    return hashlib.sha256(("\n".join(lines) + "\n").encode()).hexdigest()


def q08_library_versions():
    import PIL
    from PIL import features
    return {"python": platform.python_version(), "numpy": np.__version__, "pillow": PIL.__version__,
            "libjpeg": features.version("jpg"), "libjpeg_turbo": bool(features.check_feature("libjpeg_turbo")),
            "platform": platform.platform()}


def q08_manifest(hashes, n_images, source):
    spec = q08_spec_record()
    arrays = {}
    for key in sorted(hashes):
        entry = {"sha256": hashes[key]}
        if key not in Q08_FIXED_KEYS:
            name, severity, info = q08_key_info(key)
            entry.update(corruption=name, family=info["family"], severity=severity,
                         params=info["params"][severity - 1], library_dependent=q08_library_dependent(key))
        arrays[key] = entry
    return {"set_name": Q08_SET_NAME, "set_version": Q08_SET_VERSION, "set_sha256": q08_set_sha256(hashes),
            "spec_sha256": hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest(), "spec": spec,
            "arrays": arrays, "image_shape": [32, 32, 3], "n_images": n_images,
            "split": {"val_size": VAL_SIZE, "split_seed": SPLIT_SEED, "val_indices_sha256": int64_sha256(VAL_IDX),
                      "labels_sha256": int64_sha256(VAL_LABELS_ALL), "split_function": "stratified_split (training notebooks)"},
            "cache_fingerprint": META["fingerprint"], "libraries": q08_library_versions(), "source": source,
            "generated_by": "Archive/Q08_shiftval.ipynb",
            "created_at": datetime.now(ZoneInfo("America/New_York")).isoformat(timespec="seconds")}


def q08_compare(hashes, reference, label):
    """'exact' or 'library-mismatch'; any other difference raises (NumPy-only sets must match bit for bit)."""
    if reference.get("set_version") != Q08_SET_VERSION or reference.get("set_name") != Q08_SET_NAME:
        raise RuntimeError(f"{label} describes {reference.get('set_name')}, not {Q08_SET_NAME}")
    expected = {key: entry["sha256"] for key, entry in reference["arrays"].items()}
    missing = sorted(set(expected) ^ set(hashes))
    differing = sorted(key for key in set(expected) & set(hashes) if expected[key] != hashes[key])
    strict = [key for key in differing if not q08_library_dependent(key)]
    library = [key for key in differing if q08_library_dependent(key)]
    if missing or strict:
        raise RuntimeError(
            f"{label}: arrays differ from the locked {Q08_SET_NAME} reference: {missing + strict}. NumPy-only sets and "
            "the clean images/labels/indices must match exactly. Use the shared .npz (Q08_SOURCE='npz'); an "
            "intentional change of the set needs a new Q08_SET_VERSION.")
    if library:
        message = (f"{label}: {len(library)} Pillow-dependent arrays differ from the reference "
                   f"({', '.join(library)}). This build {q08_library_versions()} vs reference "
                   f"{reference.get('libraries')}. JPEG/pixelate results are then NOT byte-identical to the locked "
                   "set; load the shared .npz (Q08_SOURCE='npz') for exact comparability, or set "
                   "Q08_REQUIRE_EXACT=True to stop here.")
        if Q08_REQUIRE_EXACT:
            raise RuntimeError(message)
        print("!" * 100 + "\nWARNING: " + message + "\n" + "!" * 100, flush=True)
        return "library-mismatch", library
    return "exact", []


def q08_default_dir():
    if Q08_DIR:
        return Path(Q08_DIR)
    if PATHS.env == "colab":
        return drive_dir() / "q08_shiftval"  # beside data/ and cache.zip: dataset-derived, never under result/
    if PATHS.env == "kaggle":
        return work_dir(PATHS.env) / "q08_shiftval"
    return LOCAL_PROJECT_DIR / "downloads" / "q08_shiftval"  # downloads/ is git-ignored


def q08_npz_path():
    return q08_default_dir() / f"{Q08_SET_NAME}_set.npz"


def _q08_root_key(path):
    return os.path.normcase(os.path.abspath(os.path.expanduser(str(path))))


def q08_wait_for_folder(path, wait_s=0.0):
    """True once `path` is a folder. A Colab MyDrive shortcut can appear a few seconds after mounting: poll briefly."""
    deadline = time.monotonic() + max(0.0, float(wait_s))
    waiting = False
    while True:
        try:
            if path.is_dir():
                if waiting:
                    print(f"[q08] extra result root is now available: {path}")
                return True
        except OSError as error:  # e.g. a Drive mount that is still starting
            print(f"[q08] cannot access {path} yet: {error}")
        if time.monotonic() >= deadline:
            return False
        if not waiting:
            print(f"[q08] waiting up to {wait_s:g}s for {path} (a Drive shortcut may appear shortly after mounting)",
                  flush=True)
            waiting = True
        time.sleep(min(3.0, max(0.05, deadline - time.monotonic())))


_Q08_ROOTS = None


def q08_result_roots(refresh=False):
    """Search order for checkpoints and their artifacts: RESULT_DIR, then the available Q08_EXTRA_RESULT_ROOTS.

    Resolved once per session (the Colab wait happens only once). Extra roots are only ever read; every Q08 output
    is written under RESULT_DIR (PATHS.result_dir).
    """
    global _Q08_ROOTS
    if _Q08_ROOTS is not None and not refresh:
        return list(_Q08_ROOTS)
    extras = Q08_EXTRA_RESULT_ROOTS or []
    if not isinstance(extras, (list, tuple)) or not all(isinstance(r, (str, Path)) and str(r) for r in extras):
        raise ValueError("Q08_EXTRA_RESULT_ROOTS must be a list of result folder paths (or empty)")
    own = Path(PATHS.result_dir)
    roots, seen = [own], {_q08_root_key(own)}
    for extra in map(Path, extras):
        if _q08_root_key(extra) in seen:
            continue
        seen.add(_q08_root_key(extra))
        if q08_wait_for_folder(extra, Q08_EXTRA_ROOT_WAIT_S if PATHS.env == "colab" else 0.0):
            roots.append(extra)
            note = "" if (extra / "checkpoints").is_dir() else " (no checkpoints/ folder: metadata only)"
            print(f"[q08] extra result root (read-only): {extra}{note}")
        else:
            print(f"[q08] WARNING: extra result root {extra} was not found; runs that exist only there are skipped. "
                  "On Colab check the MyDrive shortcut name (Drive > Shared with me > folder > Add shortcut to My "
                  "Drive), then rerun from section 1, or edit Q08_EXTRA_RESULT_ROOTS.", flush=True)
    _Q08_ROOTS = roots
    return list(roots)


def q08_root_kind(root):
    return "RESULT_DIR" if _q08_root_key(root) == _q08_root_key(PATHS.result_dir) else "extra (read-only)"


def q08_reference_path():
    """Where this account keeps (and, if needed, writes) the reference manifest: always under RESULT_DIR."""
    return Path(PATHS.result_dir) / "manifests" / f"{Q08_SET_NAME}_set-manifest.json"


def q08_find_reference():
    """First existing reference manifest in search order (RESULT_DIR, then the extra roots), or None."""
    for root in q08_result_roots():
        path = Path(root) / "manifests" / f"{Q08_SET_NAME}_set-manifest.json"
        if path.is_file():
            return path
    return None


def q08_read_npz(path):
    """Load a shared set; every array must match the manifest stored inside the same file."""
    with np.load(path, allow_pickle=False) as z:
        if "manifest_json" not in z.files:
            raise ValueError(f"{path} is not a Q08 set export (no manifest_json)")
        manifest = json.loads(str(z["manifest_json"][()]))
        if manifest.get("set_name") != Q08_SET_NAME:
            raise ValueError(f"{path} holds {manifest.get('set_name')}, but this notebook uses {Q08_SET_NAME}")
        keys = q08_set_keys()
        if set(z.files) != set(Q08_FIXED_KEYS) | set(keys) | {"manifest_json"}:
            raise ValueError(f"{path} does not contain exactly the {Q08_SET_NAME} arrays")
        clean, labels, indices = z["clean"], z["labels"], z["val_indices"]
        stack = np.empty((len(keys),) + clean.shape, dtype=np.uint8)
        for i, key in enumerate(keys):
            stack[i] = z[key]
    hashes = q08_fingerprints(clean, labels, indices, keys, stack)
    embedded = {key: entry["sha256"] for key, entry in manifest["arrays"].items()}
    if embedded != hashes or q08_set_sha256(hashes) != manifest.get("set_sha256"):
        raise ValueError(f"{path} is damaged: its arrays differ from its own manifest")
    return manifest, clean, labels, indices, keys, stack, hashes


def q08_export_npz(path, clean, labels, indices, keys, stack, manifest):
    """Write one compressed .npz through a verified temporary copy; the previous file survives a failure."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    arrays = {"clean": clean, "labels": np.asarray(labels, dtype="<i8"), "val_indices": np.asarray(indices, dtype="<i8"),
              **{key: stack[i] for i, key in enumerate(keys)},
              "manifest_json": np.array(json.dumps(manifest, sort_keys=True))}
    partial = path.with_name(f".{path.name}.{time.time_ns()}.tmp")
    try:
        with tempfile.TemporaryDirectory(prefix=".q08-export-") as tmp:
            local = Path(tmp) / path.name
            with open(local, "wb") as f:
                np.savez_compressed(f, **arrays)
            q08_read_npz(local)
            shutil.copyfile(local, partial)
        q08_read_npz(partial)  # verify the copy at its destination before publishing it
        replace_file(partial, path)
    finally:
        partial.unlink(missing_ok=True)
    size = path.stat().st_size
    print(f"[q08] exported {path} ({size / 2**20:.1f} MiB, file sha256 {sha256_file(path)[:16]}); share this file "
          "manually. It contains dataset-derived images: never commit it to Git or copy it under result/.")
    return path

In [ ]:
if Q08_SOURCE not in ("auto", "regenerate", "npz"):
    raise ValueError("Q08_SOURCE must be auto, regenerate or npz")
if Q08_SMOKE and Q08_SOURCE == "npz":
    raise ValueError("A smoke subset is regenerated; use Q08_SOURCE='auto' or 'regenerate' with Q08_SMOKE_N > 0")
Q08_NPZ_PATH = q08_npz_path()
Q08_REFERENCE_PATH = q08_reference_path()
_use_npz = not Q08_SMOKE and (Q08_SOURCE == "npz" or (Q08_SOURCE == "auto" and Q08_NPZ_PATH.is_file()))
if _use_npz:
    if not Q08_NPZ_PATH.is_file():
        raise FileNotFoundError(f"Q08_SOURCE='npz' but {Q08_NPZ_PATH} does not exist")
    Q08_MANIFEST, _clean, _labels, _indices, Q08_KEYS, Q08_STACK, Q08_HASHES = q08_read_npz(Q08_NPZ_PATH)
    # The shared copy must be exactly this cache's split, labels and raw validation images.
    if (Q08_MANIFEST.get("cache_fingerprint") != META["fingerprint"] or not np.array_equal(_indices, VAL_IDX)
            or not np.array_equal(_labels, VAL_Y) or not np.array_equal(_clean, VAL_X)):
        raise RuntimeError(f"{Q08_NPZ_PATH} was not made from this cache's validation split")
    Q08_SOURCE_USED = f"npz {Q08_NPZ_PATH.name}"
    print(f"[q08] loaded {len(Q08_KEYS)} sets from {Q08_NPZ_PATH} ({Q08_NPZ_PATH.stat().st_size / 2**20:.1f} MiB); "
          "arrays match the manifest inside the file and this cache's clean validation images")
    del _clean, _labels, _indices
else:
    Q08_KEYS, Q08_STACK = q08_generate(VAL_X)
    Q08_HASHES = q08_fingerprints(VAL_X, VAL_Y, VAL_IDX[:Q08_N], Q08_KEYS, Q08_STACK)
    Q08_SOURCE_USED = "regenerated from the cache"
    Q08_MANIFEST = q08_manifest(Q08_HASHES, Q08_N, Q08_SOURCE_USED)
Q08_SET_SHA256 = q08_set_sha256(Q08_HASHES)
Q08_LIBRARY_DIFFS = []


def q08_event_path(folder, role, extension):
    """Dated shared Q08 artifact MM-DD-HH-mm_Q08_shiftval-v<k>_<role><ext>, numbered on collision."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    stem = f"{run_timestamp()}_{Q08_SET_NAME}_{role}"
    path, index = folder / (stem + extension), 1
    while any(folder.glob(path.stem + "*")):
        index += 1
        path = folder / f"{stem}-{index}{extension}"
    return path


if Q08_SMOKE:
    Q08_STATUS = "smoke"  # a subset is never compared with, or written as, the reference
else:
    Q08_STATUS = "unverified"
    # The reference may come from another account's result folder (extra root); it is never written there.
    Q08_REFERENCE_SOURCE = q08_find_reference()
    _reference = json.loads(Q08_REFERENCE_SOURCE.read_text()) if Q08_REFERENCE_SOURCE is not None else None
    if _reference is not None:
        Q08_STATUS, Q08_LIBRARY_DIFFS = q08_compare(Q08_HASHES, _reference, f"reference {Q08_REFERENCE_SOURCE}")
        if Q08_STATUS != "exact":
            _observed = q08_event_path(Q08_REFERENCE_PATH.parent, "observed-manifest", ".json")
            write_json_atomic(q08_manifest(Q08_HASHES, Q08_N, Q08_SOURCE_USED), _observed)
            print(f"[q08] observed fingerprints saved for diagnosis: {_observed}")
    if Q08_LOCKED_SET_SHA256 is not None and Q08_SET_SHA256 != Q08_LOCKED_SET_SHA256:
        if Q08_STATUS != "library-mismatch":
            raise RuntimeError(f"Set fingerprint {Q08_SET_SHA256} differs from Q08_LOCKED_SET_SHA256 "
                               f"{Q08_LOCKED_SET_SHA256}; use the shared .npz or the matching reference manifest")
        print("WARNING: the set fingerprint differs from Q08_LOCKED_SET_SHA256 only through the Pillow-dependent "
              "arrays listed above")
    if _reference is None:
        Q08_REFERENCE_PATH.parent.mkdir(parents=True, exist_ok=True)
        write_json_atomic(Q08_MANIFEST, Q08_REFERENCE_PATH)
        Q08_STATUS = "locked" if Q08_LOCKED_SET_SHA256 == Q08_SET_SHA256 else "new-reference"
        print(f"[q08] no reference manifest yet: wrote {Q08_REFERENCE_PATH}. Keep it with the results (sync it to "
              "result/manifests in Git); later regenerations are checked against it.")
    else:
        if _q08_root_key(Q08_REFERENCE_SOURCE) != _q08_root_key(Q08_REFERENCE_PATH):
            Q08_REFERENCE_PATH.parent.mkdir(parents=True, exist_ok=True)
            write_json_atomic(_reference, Q08_REFERENCE_PATH)  # the other account's reference, copied unchanged
            print(f"[q08] reference manifest {Q08_REFERENCE_SOURCE} ({q08_root_kind(Q08_REFERENCE_SOURCE.parents[1])}) "
                  f"copied unchanged to {Q08_REFERENCE_PATH}, so both accounts keep the identical reference")
        if Q08_STATUS == "exact" and Q08_LOCKED_SET_SHA256 == Q08_SET_SHA256:
            Q08_STATUS = "locked"
if Q08_EXPORT_NPZ:
    if Q08_SMOKE:
        print("[q08] Q08_EXPORT_NPZ skipped: a smoke subset is never exported")
    elif _use_npz:
        print(f"[q08] Q08_EXPORT_NPZ skipped: the set was loaded from {Q08_NPZ_PATH}")
    else:
        q08_export_npz(Q08_NPZ_PATH, VAL_X, VAL_Y, VAL_IDX, Q08_KEYS, Q08_STACK, Q08_MANIFEST)
print(f"[q08] {Q08_SET_NAME}: {len(Q08_KEYS)} sets x {Q08_N:,} images + clean | source {Q08_SOURCE_USED} | "
      f"status {Q08_STATUS} | set sha256 {Q08_SET_SHA256}")
print(f"      libraries {q08_library_versions()}")
if not Q08_SMOKE and Q08_LOCKED_SET_SHA256 is None:
    print(f"      To lock this set after the one-time severity check, set Q08_LOCKED_SET_SHA256 = {Q08_SET_SHA256!r}")

# Visual check: four validation images under every corruption at severity 3.
_examples = [int(np.flatnonzero(VAL_Y == c)[0]) for c in range(NUM_CLASSES) if np.any(VAL_Y == c)][:4]
_names = list(Q08_CORRUPTIONS)
_fig, _axes = plt.subplots(len(_examples), len(_names) + 1, figsize=(1.3 * (len(_names) + 1), 1.4 * len(_examples)),
                           layout="constrained", squeeze=False)
for _row, _pos in enumerate(_examples):
    _axes[_row, 0].imshow(VAL_X[_pos])
    for _col, _name in enumerate(_names, start=1):
        _axes[_row, _col].imshow(Q08_STACK[Q08_KEYS.index(q08_set_key(_name, 3)), _pos])
for _col, _title in enumerate(["clean"] + _names):
    _axes[0, _col].set_title(_title.replace("_", "\n") + ("" if _col == 0 else "\ns3"), fontsize=7)
for _ax in _axes.flat:
    _ax.axis("off")
plt.show()
plt.close(_fig)

## 5. Model definitions (verbatim copies)

Each cell below stores, as text, the two section-4 code cells of one training notebook (model definition and
parameter-budget/build cell), unchanged. They are executed in a separate namespace per family, so the six models
keep their own `MODEL`, `make_model`, `ARCH_DEFAULTS` and switches without overwriting each other, and the copied
checkpoint helpers (`model_fingerprint`, `validate_checkpoint`, `load_model`) run against that family. A
checkpoint is therefore rebuilt from its **saved** architecture options with the same code, parameter count and
model fingerprint as in its training notebook. **Do not edit these copies:** `tests/test_notebooks.py` fails when a
copy differs from its training notebook; after a model-cell change there, copy the new cell text here unchanged.

In [ ]:
# Verbatim copy of A_resnet.ipynb section 4: its model cell, then its parameter-budget cell.
# Do not edit: tests/test_notebooks.py fails when this text differs from the training notebook.
MODEL_SOURCES = {}  # family -> (model cell, parameter-budget cell), filled by the six cells

MODEL_SOURCES['A'] = (
r'''MODEL = "A"
MODEL_DESCRIPTION = "ResNet BasicBlock [64,128,256] x [3,5,3]"
EXPECTED_PARAMS = 4_918_602


class ECAModule(nn.Module):
    """Spatial average -> shared 1D convolution across channels -> sigmoid channel gates."""
    def __init__(self, kernel_size=3):
        super().__init__()
        self.conv = nn.Conv1d(1, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        weights = torch.sigmoid(self.conv(x.mean(dim=(2, 3)).unsqueeze(1))).squeeze(1)
        return x * weights[:, :, None, None]


class BasicBlock(nn.Module):
    """conv-BN-ReLU-conv-BN, added to the shortcut, then ReLU (He et al., 2016)."""

    def __init__(self, in_ch, out_ch, stride=1, channel_attention="none", eca_kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.eca = ECAModule(eca_kernel_size) if channel_attention == "eca" else None
        if stride == 1 and in_ch == out_ch:
            self.shortcut = nn.Identity()
        else:  # 1x1 projection shortcut where the shape changes
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.eca is not None:
            out = self.eca(out)
        return F.relu(out + self.shortcut(x))


class PreActBasicBlock(nn.Module):
    """Full pre-activation: raw identity; projections use BN-ReLU input; no post-add ReLU."""
    def __init__(self, in_ch, out_ch, stride=1, channel_attention="none", eca_kernel_size=3):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.eca = ECAModule(eca_kernel_size) if channel_attention == "eca" else None
        self.shortcut = (nn.Identity() if stride == 1 and in_ch == out_ch else
                         nn.Conv2d(in_ch, out_ch, 1, stride, bias=False))

    def forward(self, x):
        pre = F.relu(self.bn1(x))
        shortcut = x if isinstance(self.shortcut, nn.Identity) else self.shortcut(pre)
        out = self.conv1(pre)
        out = self.conv2(F.relu(self.bn2(out)))
        if self.eca is not None:
            out = self.eca(out)
        return shortcut + out


class ResNet(nn.Module):
    """3x3 stem (no max-pool), one stage per entry of `channels`, global average pool, linear head."""

    def __init__(self, channels=(64, 128, 256), blocks=(3, 5, 3), num_classes=10, channel_attention="none", eca_kernel_size=3, activation_order="post"):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels[0], 3, 1, 1, bias=False), nn.BatchNorm2d(channels[0]), nn.ReLU(inplace=True)
        )
        if activation_order == "pre":
            # Preserve the original post-activation modules/state keys when the switch is off.
            self.stem = nn.Sequential(nn.Conv2d(3, channels[0], 3, 1, 1, bias=False))
            self.final_norm = nn.Sequential(nn.BatchNorm2d(channels[-1]), nn.ReLU())
        block = PreActBasicBlock if activation_order == "pre" else BasicBlock
        stages, in_ch = [], channels[0]
        for i, (ch, n) in enumerate(zip(channels, blocks)):
            layers = []
            for j in range(n):
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(block(in_ch, ch, stride, channel_attention, eca_kernel_size))
                in_ch = ch
            stages.append(nn.Sequential(*layers))
        self.stages = nn.Sequential(*stages)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = self.stages(self.stem(x))
        if hasattr(self, "final_norm"):
            x = self.final_norm(x)
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


ARCH_DEFAULTS = {"implementation_version": 1, "channel_attention": "none", "eca_kernel_size": 3}


ARCH_OPTION_DEFAULTS = {"depth_preset": "baseline", "capacity_preset": "baseline", "activation_order": "post"}  # Omitted baseline key preserves old checkpoint identities.


def architecture_from_config():
    return {"channel_attention": CHANNEL_ATTENTION, "eca_kernel_size": ECA_KERNEL_SIZE,
            "depth_preset": DEPTH_PRESET, "capacity_preset": CAPACITY_PRESET, "activation_order": ACTIVATION_ORDER}


def architecture_channels(options):
    return (64, 160, 224) if options.get("capacity_preset") == "mid" else (64, 128, 256)


def architecture_blocks(options):
    if options.get("capacity_preset") == "mid":
        return (3, 4, 3)
    return (4, 5, 3) if options.get("depth_preset", "baseline") == "early" else (3, 5, 3)


def validate_architecture(options):
    if options.get("activation_order", "post") not in ("post", "pre"):
        raise ValueError("ACTIVATION_ORDER must be post or pre")
    capacity = options.get("capacity_preset", "baseline")
    if capacity not in ("baseline", "mid"):
        raise ValueError("CAPACITY_PRESET must be baseline or mid")
    if capacity == "mid" and (options.get("depth_preset", "baseline") != "baseline" or options["channel_attention"] != "none"):
        raise ValueError("T26 mid capacity requires DEPTH_PRESET=baseline and CHANNEL_ATTENTION=none")
    depth = options.get("depth_preset", "baseline")
    if depth not in ("baseline", "early"):
        raise ValueError("DEPTH_PRESET must be baseline or early")
    if options["channel_attention"] not in ("none", "eca"):
        raise ValueError("CHANNEL_ATTENTION must be 'none' or 'eca'")
    k = options["eca_kernel_size"]
    if type(k) is not int or not 1 <= k <= 63 or k % 2 != 1:
        raise ValueError("ECA_KERNEL_SIZE must be a positive odd integer <= 63; T48 uses 3")


def describe_architecture(options):
    if options.get("activation_order", "post") == "pre":
        return (f"PreAct ResNet {list(architecture_channels(options))} x {list(architecture_blocks(options))}; "
                "BN-ReLU-conv branches, raw identity / preactivated projection, final BN-ReLU" +
                (f"; ECA k={options['eca_kernel_size']}" if options['channel_attention'] == 'eca' else ''))
    if options.get("capacity_preset") == "mid":
        return "ResNet BasicBlock [64,160,224] x [3,4,3]"
    if options.get("depth_preset") == "early":
        return "ResNet BasicBlock [64,128,256] x [4,5,3]" + (f"; ECA k={options['eca_kernel_size']} after each bn2" if options["channel_attention"] == "eca" else "")
    return MODEL_DESCRIPTION + (f"; ECA k={options['eca_kernel_size']} after each bn2" if options["channel_attention"] == "eca" else "")


def architecture_feature(options):
    if options.get("activation_order", "post") == "pre":
        return "preact"
    if options.get("capacity_preset") == "mid":
        return "mid"
    if options.get("depth_preset") == "early":
        return "early-eca" if options["channel_attention"] == "eca" else "early"
    return "eca" if options["channel_attention"] == "eca" else None


def expected_param_count(options):
    if options.get("capacity_preset") == "mid":
        return 4_563_594 - (2 * sum(architecture_channels(options)[1:]) if options.get("activation_order", "post") == "pre" else 0)
    base = 4_992_586 if options.get("depth_preset") == "early" else EXPECTED_PARAMS
    eca_params = sum(architecture_blocks(options)) * options["eca_kernel_size"] if options["channel_attention"] == "eca" else 0
    return base + eca_params - (2 * sum(architecture_channels(options)[1:]) if options.get("activation_order", "post") == "pre" else 0)


def make_model(num_classes=10, architecture=None):
    options = resolve_architecture(architecture)
    return ResNet(architecture_channels(options), architecture_blocks(options), num_classes=num_classes,
                  channel_attention=options["channel_attention"], eca_kernel_size=options["eca_kernel_size"],
                  activation_order=options.get("activation_order", "post"))''',
r'''PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - (set(ARCH_DEFAULTS) | set(ARCH_OPTION_DEFAULTS))
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = {**ARCH_DEFAULTS, **ARCH_OPTION_DEFAULTS, **options}
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    # Canonical omission preserves fingerprints/topology records for all old options.
    return {k: v for k, v in resolved.items() if k not in ARCH_OPTION_DEFAULTS or v != ARCH_OPTION_DEFAULTS[k]}


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = (WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else
                architecture_channels(details) if MODEL == "A" else (64, 128, 256))
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=list(architecture_blocks(details)),
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model''',
)


In [ ]:
# Verbatim copy of B_wrn.ipynb section 4: its model cell, then its parameter-budget cell.
# Do not edit: tests/test_notebooks.py fails when this text differs from the training notebook.
MODEL_SOURCES['B'] = (
r'''MODEL = "B"
MODEL_DESCRIPTION = "WRN-22-4 (pre-activation, widths 16/64/128/256)"
EXPECTED_PARAMS = 4_298_970


class PreActBlock(nn.Module):
    """Pre-activation block BN-ReLU-conv-BN-ReLU-conv (He et al., 2016b), as used in WRN."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        # WRN uses a plain 1x1 conv (no BN) on the pre-activated input when the shape changes.
        self.shortcut = None if (stride == 1 and in_ch == out_ch) else nn.Conv2d(in_ch, out_ch, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = x if self.shortcut is None else self.shortcut(out)
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class WideResNet(nn.Module):
    """WRN-depth-k: widths 16 / 16k / 32k / 64k, (depth-4)/6 blocks per group."""

    def __init__(self, depth=22, widen=4, num_classes=10, stage_channels=None):
        super().__init__()
        assert (depth - 4) % 6 == 0, "WRN depth must be 6n+4"
        n = (depth - 4) // 6
        widths = [16, *(stage_channels if stage_channels is not None else (16 * widen, 32 * widen, 64 * widen))]
        self.stem = nn.Conv2d(3, widths[0], 3, 1, 1, bias=False)
        groups, in_ch = [], widths[0]
        for g in range(3):
            layers = []
            for j in range(n):
                stride = 2 if (g > 0 and j == 0) else 1
                layers.append(PreActBlock(in_ch, widths[g + 1], stride))
                in_ch = widths[g + 1]
            groups.append(nn.Sequential(*layers))
        self.groups = nn.Sequential(*groups)
        self.bn = nn.BatchNorm2d(in_ch)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = F.relu(self.bn(self.groups(self.stem(x))))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


WIDTH_PRESETS = {"baseline": (64, 128, 256), "wide68": (68, 136, 272), "deep56": (56, 112, 224), "balanced": (64, 144, 272), "shallow80": (80, 160, 320)}
WIDTH_DEPTHS = {"baseline": 22, "wide68": 22, "deep56": 28, "balanced": 22, "shallow80": 16}
ARCH_OPTION_DEFAULTS = {}
ARCH_DEFAULTS = {"implementation_version": 1, "width_preset": "baseline"}


def architecture_from_config():
    return {"width_preset": WIDTH_PRESET}


def architecture_blocks(options):
    return ((WIDTH_DEPTHS[options["width_preset"]] - 4) // 6,) * 3


def validate_architecture(options):
    if not isinstance(options["width_preset"], str) or options["width_preset"] not in WIDTH_PRESETS:
        raise ValueError("WIDTH_PRESET must be baseline, wide68, deep56, balanced or shallow80")


def describe_architecture(options):
    if options["width_preset"] == "shallow80":
        return "WRN-16-5 (pre-activation, widths 16/80/160/320, blocks 2/2/2)"
    if options["width_preset"] == "balanced":
        return "WRN-22 (pre-activation, widths 16/64/144/272, blocks 3/3/3)"
    if options["width_preset"] == "deep56":
        return "WRN-28 (pre-activation, widths 16/56/112/224, blocks 4/4/4)"
    return MODEL_DESCRIPTION if options["width_preset"] == "baseline" else "WRN-22 (pre-activation, widths 16/68/136/272, blocks 3/3/3)"


def architecture_feature(options):
    return {"baseline": None, "wide68": "wide", "deep56": "deep", "balanced": "balanced", "shallow80": "shallow"}[options["width_preset"]]


def expected_param_count(options):
    return {"baseline": EXPECTED_PARAMS, "wide68": 4_851_866, "deep56": 4_480_474, "balanced": 4_949_754, "shallow80": 4_289_754}[options["width_preset"]]


def make_model(num_classes=10, architecture=None):
    options = resolve_architecture(architecture)
    return WideResNet(depth=WIDTH_DEPTHS[options["width_preset"]], num_classes=num_classes,
                      stage_channels=WIDTH_PRESETS[options["width_preset"]])''',
r'''PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - (set(ARCH_DEFAULTS) | set(ARCH_OPTION_DEFAULTS))
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = {**ARCH_DEFAULTS, **ARCH_OPTION_DEFAULTS, **options}
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    # Canonical omission preserves fingerprints/topology records for all old options.
    return {k: v for k, v in resolved.items() if k not in ARCH_OPTION_DEFAULTS or v != ARCH_OPTION_DEFAULTS[k]}


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = (WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else
                architecture_channels(details) if MODEL == "A" else (64, 128, 256))
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=list(architecture_blocks(details)),
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model''',
)

In [ ]:
# Verbatim copy of C_seresnet.ipynb section 4: its model cell, then its parameter-budget cell.
# Do not edit: tests/test_notebooks.py fails when this text differs from the training notebook.
MODEL_SOURCES['C'] = (
r'''MODEL = "C"
MODEL_DESCRIPTION = "SE-ResNet BasicBlock [64,128,256] x [3,5,3], SE r=16"
EXPECTED_PARAMS = 4_956_654


class SEModule(nn.Module):
    """Squeeze-and-Excitation channel gating (Hu et al., 2018)."""

    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        w = x.mean(dim=(2, 3))
        w = torch.sigmoid(self.fc2(F.relu(self.fc1(w))))
        return x * w[:, :, None, None]


class InstanceBatchNorm2d(nn.Module):
    """IBN-a: fixed half-channel affine IN + BN; IN uses per-image stats in train and eval.

    Stage placement is controlled by SEResNet, not by train/eval mode.
    Reference: https://github.com/XingangPan/IBN-Net/blob/master/ibnnet/modules.py
    """

    def __init__(self, channels):
        super().__init__()
        if type(channels) is not int or channels < 2 or channels % 2:
            raise ValueError("Half-channel IBN requires a positive even channel count >= 2")
        self.half = channels // 2
        self.instance = nn.InstanceNorm2d(self.half, affine=True, track_running_stats=False)
        self.batch = nn.BatchNorm2d(self.half)

    def forward(self, x):
        left, right = x.split(self.half, dim=1)
        return torch.cat((self.instance(left.contiguous()), self.batch(right.contiguous())), dim=1)


class SEBasicBlock(nn.Module):
    """conv-(BN or IBN)-ReLU-conv-BN-SE, shortcut addition, then ReLU."""

    def __init__(self, in_ch, out_ch, stride=1, reduction=16, ibn=False):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = InstanceBatchNorm2d(out_ch) if ibn else nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.se = SEModule(out_ch, reduction)
        if stride == 1 and in_ch == out_ch:
            self.shortcut = nn.Identity()
        else:  # 1x1 projection shortcut where the shape changes
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.se(self.bn2(self.conv2(out)))
        return F.relu(out + self.shortcut(x))


class SEResNet(nn.Module):
    """3x3 stem (no max-pool), one stage of SE blocks per entry of `channels`, global average pool, linear head."""

    def __init__(self, channels=(64, 128, 256), blocks=(3, 5, 3), reduction=16, num_classes=10, ibn_stage1=False):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels[0], 3, 1, 1, bias=False), nn.BatchNorm2d(channels[0]), nn.ReLU(inplace=True)
        )
        stages, in_ch = [], channels[0]
        for i, (ch, n) in enumerate(zip(channels, blocks)):
            layers = []
            for j in range(n):
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(SEBasicBlock(in_ch, ch, stride, reduction, ibn=ibn_stage1 and i == 0))
                in_ch = ch
            stages.append(nn.Sequential(*layers))
        self.stages = nn.Sequential(*stages)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = self.stages(self.stem(x))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


ARCH_DEFAULTS = {"implementation_version": 1, "se_reduction": 16}


ARCH_OPTION_DEFAULTS = {"ibn_stage1": False}


def architecture_blocks(options):
    return (3, 5, 3)


def architecture_from_config():
    return {"se_reduction": SE_REDUCTION, "ibn_stage1": IBN_STAGE1}


def validate_architecture(options):
    if type(options["ibn_stage1"]) is not bool:
        raise ValueError("IBN_STAGE1 must be True or False; T57 uses stage 1 only, fixed IN ratio 0.5")
    if type(options["se_reduction"]) is not int or options["se_reduction"] not in (16, 8):
        raise ValueError("SE_REDUCTION must be 16 (baseline) or 8 (T06)")


def describe_architecture(options):
    description = MODEL_DESCRIPTION.replace("r=16", f"r={options['se_reduction']}")
    if options.get("ibn_stage1", False):
        description += "; stage1 conv1 IBN-a (half affine IN/half BN, IN per-image stats)"
    return description


def architecture_feature(options):
    if options.get("ibn_stage1", False):
        return "ibn" if options["se_reduction"] == 16 else "se8ibn"
    return "se8" if options["se_reduction"] == 8 else None


def expected_param_count(options):
    return EXPECTED_PARAMS if options["se_reduction"] == 16 else 4_993_106


def make_model(num_classes=10, architecture=None):
    options = resolve_architecture(architecture)
    return SEResNet((64, 128, 256), (3, 5, 3), reduction=options["se_reduction"],
                    num_classes=num_classes, ibn_stage1=options.get("ibn_stage1", False))''',
r'''PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero affine BN/IN; original BN initialization unchanged."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and m.affine:
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, normalization gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - (set(ARCH_DEFAULTS) | set(ARCH_OPTION_DEFAULTS))
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = {**ARCH_DEFAULTS, **ARCH_OPTION_DEFAULTS, **options}
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    # Canonical omission preserves fingerprints/topology records for all old options.
    return {k: v for k, v in resolved.items() if k not in ARCH_OPTION_DEFAULTS or v != ARCH_OPTION_DEFAULTS[k]}


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = (WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else
                architecture_channels(details) if MODEL == "A" else (64, 128, 256))
    if MODEL == "C" and details.get("ibn_stage1", False):
        details["ibn"] = {"stages": [1], "placement": "after conv1, before ReLU in each block",
                          "in_ratio": 0.5, "in_affine": True, "in_track_running_stats": False,
                          "implementation_version": 1}
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=list(architecture_blocks(details)),
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model''',
)

In [ ]:
# Verbatim personal E model/build cells; training-only mask attributes do not change evaluation identities.
MODEL_SOURCES['E'] = (
r'''MODEL = "E"
MODEL_DESCRIPTION = "PyramidNet-110 basic, alpha=98 (widths 16->114)"
EXPECTED_PARAMS = 4_963_991


class DropPath(nn.Module):
    """Stochastic depth (Huang et al., 2016) for one residual branch, per sample.

    In training, each image's branch is dropped with probability p and survivors are scaled by 1 / (1 - p);
    in evaluation it is the identity. The mask comes from the global torch RNG of the input's device.
    """

    def __init__(self, p=0.0):
        super().__init__()
        self.p = float(p)
        self._sam_mask = None  # transient, never part of checkpoints or model identity

    def forward(self, x):
        if not self.training or self.p == 0.0:
            return x
        keep = 1.0 - self.p
        mask = self._sam_mask
        if mask is None:
            mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep
        return x * mask.to(x.dtype) / keep

    def extra_repr(self):
        return f"p={self.p:.4f}"


MIXSTYLE_AFTER_STAGES = (1, 2)  # training-only MixStyle (MIXSTYLE, section 1) mixes the outputs of stages 1 and 2


def mix_feature_styles(x, apply, lam, perm, eps):
    """MixStyle (Zhou et al., ICLR 2021), "random" variant, on one training batch; parameter-free.

    Each image's per-channel mean mu and standard deviation sigma over the spatial positions (unbiased variance + eps,
    computed in float32 and detached, as in the reference code) are replaced by lam * stat + (1 - lam) * stat of its
    partner image perm[i]; the result is cast back to x's dtype. `apply` (0-dim bool tensor) selects the mixed batch
    or exactly x; lam has shape (B, 1, 1, 1). The draws come from the trainer's MixStyleSampler (section 5).
    """
    x32 = x.float()
    mu = x32.mean(dim=(2, 3), keepdim=True).detach()
    sigma = (x32.var(dim=(2, 3), keepdim=True) + eps).sqrt().detach()
    mixed_mu = lam * mu + (1 - lam) * mu[perm]
    mixed_sigma = lam * sigma + (1 - lam) * sigma[perm]
    mixed = ((x32 - mu) / sigma * mixed_sigma + mixed_mu).to(x.dtype)
    return torch.where(apply, mixed, x)


class PyramidBlock(nn.Module):
    """BN-conv-BN-ReLU-conv-BN (Han et al., 2017), added to a parameter-free shortcut; no ReLU after the addition."""

    def __init__(self, in_ch, out_ch, stride=1, drop_path=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)
        self.drop_path = DropPath(drop_path)
        # Shortcut: identity, 2x2 average pool where the block downsamples, zero padding for the extra channels.
        self.pool = nn.AvgPool2d(2, 2, ceil_mode=True) if stride != 1 else nn.Identity()
        self.extra_ch = out_ch - in_ch

    def forward(self, x):
        out = self.conv1(self.bn1(x))
        out = self.conv2(F.relu(self.bn2(out)))
        out = self.drop_path(self.bn3(out))
        shortcut = self.pool(x)
        if self.extra_ch:
            shortcut = F.pad(shortcut, (0, 0, 0, 0, 0, self.extra_ch))
        return out + shortcut


class PyramidNet(nn.Module):
    """3x3 stem (16 channels, BN), 3 stages of (depth-2)/6 pyramidal blocks, BN-ReLU, global average pool, linear.

    The width grows by alpha / N after each of the N blocks. As in the reference implementation, the width is
    accumulated as a float and each block uses int(round(width)) channels. Block l of N drops its residual branch
    with probability drop_path * l / N (linear decay rule of Huang et al., 2016). A training forward may receive
    MixStyle draws for the points after stages MIXSTYLE_AFTER_STAGES; this adds no module, parameter or buffer.
    """

    mixstyle_layers = len(MIXSTYLE_AFTER_STAGES)  # number of MixStyle draws a training forward accepts

    def __init__(self, depth=110, alpha=98, drop_path=0.0, num_classes=10, head="baseline"):
        super().__init__()
        assert (depth - 2) % 6 == 0, "basic-block PyramidNet depth must be 6n+2"
        n = (depth - 2) // 6
        total = 3 * n
        add_rate = alpha / total
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16))
        stages, in_ch, width, index = [], 16, 16.0, 0
        for i in range(3):
            layers = []
            for j in range(n):
                index += 1
                width += add_rate
                out_ch = int(round(width))
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(PyramidBlock(in_ch, out_ch, stride, drop_path * index / total))
                in_ch = out_ch
            stages.append(nn.Sequential(*layers))
        self.stages = nn.Sequential(*stages)
        self.bn = nn.BatchNorm2d(in_ch)
        self.head = head
        if head == "head256":
            self.neck = nn.Sequential(nn.Conv2d(in_ch, 256, 1, bias=False),
                                      nn.BatchNorm2d(256), nn.ReLU())
        if head == "multistage":
            self.stage2_bn = nn.BatchNorm2d(81)
        self.fc = nn.Linear(256 if head == "head256" else (81 + in_ch if head == "multistage" else in_ch), num_classes)

    def forward(self, x, mixstyle=None):
        """`mixstyle` = (apply, lam, perm, eps) from MixStyleSampler.draw, or None. Without draws, and always in
        evaluation mode, this is exactly the plain PyramidNet forward."""
        x = self.stem(x)
        use_mixstyle = mixstyle is not None and self.training
        if not use_mixstyle and self.head != "multistage":
            x = self.stages(x)
        else:
            if use_mixstyle:
                apply, lam, perm, eps = mixstyle
            layer = 0
            for index, stage in enumerate(self.stages, 1):
                x = stage(x)
                if use_mixstyle and index in MIXSTYLE_AFTER_STAGES:
                    x = mix_feature_styles(x, apply[layer], lam[layer], perm[layer], eps)
                    layer += 1
                if self.head == "multistage" and index == 2:
                    # Separate out-of-place BN/ReLU branch; stage3 still receives unmodified x.
                    stage2 = torch.flatten(F.adaptive_avg_pool2d(F.relu(self.stage2_bn(x)), 1), 1)
        x = F.relu(self.bn(x))
        if self.head == "head256":
            x = self.neck(x)
        x = torch.flatten(F.adaptive_avg_pool2d(x, 1), 1)
        if self.head == "multistage":
            x = torch.cat((stage2, x), dim=1)
        return self.fc(x)


ARCH_DEFAULTS = {"implementation_version": 1, "drop_path": 0.0}


def architecture_from_config():
    return {"drop_path": DROP_PATH, "head": HEAD}


def validate_architecture(options):
    if options.get("head", "baseline") not in ("baseline", "head256", "multistage"):
        raise ValueError("HEAD must be baseline, head256 or multistage")
    rate = options["drop_path"]
    if isinstance(rate, bool) or not isinstance(rate, (int, float)) or not 0 <= rate < 1:
        raise ValueError(f"DROP_PATH = {rate!r}; choose a rate in [0, 1) (0 = no stochastic depth)")


def describe_architecture(options):
    # The rate is saved with the options and enters the architecture fingerprint: resuming with another rate is
    # rejected, and evaluation rebuilds the saved rate whatever the current DROP_PATH.
    description = f"{MODEL_DESCRIPTION}, stochastic depth DROP_PATH={float(options['drop_path'])!r}"
    if options.get("head") == "multistage":
        return description + ", pooled stage2+3 head (81+114 channels)"
    return description + (", pre-GAP Conv1x1/BN/ReLU head256" if options.get("head") == "head256" else "")


def architecture_feature(options):
    if options.get("head") == "multistage":
        return "multistage"
    if options.get("head") == "head256":
        return "head256"
    return "stochdepth" if options["drop_path"] > 0 else None


def expected_param_count(options):
    return {"baseline": EXPECTED_PARAMS, "head256": 4_995_107, "multistage": 4_964_963}[options.get("head", "baseline")]


def architecture_topology(options):
    """Stem width, output channels of each stage (the width grows by alpha/54 in every block) and blocks per stage."""
    topology = {"stem_channels": 16, "stage_channels": [49, 81, 114], "blocks": [18, 18, 18]}
    if options.get("head") == "head256":
        topology.update(head_channels=256, head_position="before_global_average_pool")
    if options.get("head") == "multistage":
        topology.update(head_channels=195, pooled_stages=[2, 3], stage2_head_bn=True)
    return topology


def make_model(num_classes=10, architecture=None):
    options = resolve_architecture(architecture)
    return PyramidNet(depth=110, alpha=98, drop_path=options["drop_path"], num_classes=num_classes, head=options.get("head", "baseline"))''',
r'''PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - set(ARCH_DEFAULTS) - {"head"}
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = dict(ARCH_DEFAULTS, **options)
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    # Omit the default additive option so legacy fingerprints and resumes stay identical.
    if resolved.get("head") == "baseline":
        resolved.pop("head")
    return resolved


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else (64, 128, 256)
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=[3, 3, 3] if MODEL == "B" else [3, 5, 3],
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model''',
)


In [ ]:
# Verbatim copy of F_shake.ipynb section 4: its model cell, then its parameter-budget cell.
# Do not edit: tests/test_notebooks.py fails when this text differs from the training notebook.
MODEL_SOURCES['F'] = (
r'''MODEL = "F"
MODEL_DESCRIPTION = "Shake-Shake-26 2x40d (widths 40/80/160), Shake-Shake-Image: forward and backward per image"
EXPECTED_PARAMS = 4_561_354


class ShakeShake(torch.autograd.Function):
    """Forward alpha * a + (1 - alpha) * b; backward passes beta and 1 - beta of the gradient (Gastaldi, 2017)."""

    @staticmethod
    def forward(ctx, a, b, alpha, beta):
        ctx.save_for_backward(beta)
        return torch.lerp(b, a, alpha.to(a.dtype))  # = alpha * a + (1 - alpha) * b

    @staticmethod
    def backward(ctx, grad):
        (beta,) = ctx.saved_tensors
        return grad * beta.to(grad.dtype), grad * (1 - beta).to(grad.dtype), None, None


def shake_shake(a, b, training):
    """Combine two residual branches: independent per-image U(0,1) coefficients for the forward (alpha) and
    backward (beta) pass in training, drawn from the global torch RNG of the input's device; 0.5 in evaluation."""
    if not training:
        return 0.5 * (a + b)
    alpha, beta = torch.rand(2, a.shape[0], 1, 1, 1, device=a.device)
    return ShakeShake.apply(a, b, alpha, beta)


def shake_branch(in_ch, out_ch, stride):
    """ReLU-conv3x3-BN-ReLU-conv3x3-BN residual branch."""
    return nn.Sequential(
        nn.ReLU(), nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False), nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True), nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False), nn.BatchNorm2d(out_ch)
    )


class ShakeShortcut(nn.Module):
    """Shortcut where the shape changes: ReLU, then two 1x1 convs (out/2 channels each) after a stride-s 1x1
    average pool, the second on the input shifted by one pixel so that it sees the other pixels; concat, BN."""

    def __init__(self, in_ch, out_ch, stride):
        super().__init__()
        self.stride = stride
        self.conv1 = nn.Conv2d(in_ch, out_ch // 2, 1, bias=False)
        self.conv2 = nn.Conv2d(in_ch, out_ch // 2, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        x = F.relu(x)
        y1 = self.conv1(F.avg_pool2d(x, 1, self.stride))
        y2 = self.conv2(F.avg_pool2d(F.pad(x[:, :, 1:, 1:], (0, 1, 0, 1)), 1, self.stride))
        return self.bn(torch.cat([y1, y2], 1))

    def extra_repr(self):
        return f"stride={self.stride}"


class ShakeBlock(nn.Module):
    """Two residual branches combined by shake_shake and added to the shortcut; no ReLU after the addition."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.branch1 = shake_branch(in_ch, out_ch, stride)
        self.branch2 = shake_branch(in_ch, out_ch, stride)
        if stride == 1 and in_ch == out_ch:
            self.shortcut = nn.Identity()
        else:
            self.shortcut = ShakeShortcut(in_ch, out_ch, stride)

    def forward(self, x):
        return self.shortcut(x) + shake_shake(self.branch1(x), self.branch2(x), self.training)


class ShakeShakeNet(nn.Module):
    """3x3 stem (16 channels, BN), 3 stages of (depth-2)/6 Shake-Shake blocks with widths k/2k/4k (k even),
    ReLU, global average pool, linear."""

    def __init__(self, depth=26, width=40, num_classes=10):
        super().__init__()
        assert (depth - 2) % 6 == 0, "Shake-Shake depth must be 6n+2"
        if isinstance(width, bool) or not isinstance(width, int) or width < 2 or width % 2:
            raise ValueError(f"Shake-Shake width = {width!r}; choose an even positive integer: the first stage's "
                             "two-path shortcut concatenates two halves of width/2 channels")
        n = (depth - 2) // 6
        self.stem = nn.Sequential(nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16))
        stages, in_ch = [], 16
        for i, ch in enumerate((width, 2 * width, 4 * width)):
            layers = []
            for j in range(n):
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(ShakeBlock(in_ch, ch, stride))
                in_ch = ch
            stages.append(nn.Sequential(*layers))
        self.stages = nn.Sequential(*stages)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = F.relu(self.stages(self.stem(x)))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


ARCH_DEFAULTS = {"implementation_version": 1}  # scheme F has no architecture switches


def architecture_from_config():
    return {}


def validate_architecture(options):
    pass  # resolve_architecture already rejects unknown options and other implementation versions


def describe_architecture(options):
    return MODEL_DESCRIPTION


def architecture_feature(options):
    return None


def expected_param_count(options):
    return EXPECTED_PARAMS


def architecture_topology(options):
    """Stem width, output channels and number of blocks of each stage (saved with the architecture options)."""
    return {"stem_channels": 16, "stage_channels": [40, 80, 160], "blocks": [4, 4, 4]}


def make_model(num_classes=10, architecture=None):
    resolve_architecture(architecture)
    return ShakeShakeNet(depth=26, width=40, num_classes=num_classes)''',
r'''PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - set(ARCH_DEFAULTS)
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = dict(ARCH_DEFAULTS, **options)
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    return resolved


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else (64, 128, 256)
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=[3, 3, 3] if MODEL == "B" else [3, 5, 3],
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model''',
)

In [ ]:
# Verbatim copy of G_resnext.ipynb section 4: its model cell, then its parameter-budget cell.
# Do not edit: tests/test_notebooks.py fails when this text differs from the training notebook.
MODEL_SOURCES['G'] = (
r'''MODEL = "G"
MODEL_DESCRIPTION = "ResNeXt-29 32x4d (cardinality 32, group widths 128/256/512, outputs 256/512/1024)"
EXPECTED_PARAMS = 4_775_754


class ResNeXtBlock(nn.Module):
    """1x1 conv-BN-ReLU, grouped 3x3 conv-BN-ReLU, 1x1 conv (2x channels)-BN, added to the shortcut, then ReLU."""

    expansion = 2

    def __init__(self, in_ch, cardinality=32, bottleneck_width=4, stride=1):
        super().__init__()
        group_width = cardinality * bottleneck_width
        out_ch = self.expansion * group_width
        self.conv1 = nn.Conv2d(in_ch, group_width, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(group_width)
        self.conv2 = nn.Conv2d(group_width, group_width, 3, stride, 1, groups=cardinality, bias=False)
        self.bn2 = nn.BatchNorm2d(group_width)
        self.conv3 = nn.Conv2d(group_width, out_ch, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)
        if stride == 1 and in_ch == out_ch:
            self.shortcut = nn.Identity()
        else:  # 1x1 projection shortcut where the shape changes
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        return F.relu(out + self.shortcut(x))


class ResNeXt(nn.Module):
    """3x3 stem (64 channels, no max-pool), one stage of bottleneck blocks per entry of `blocks`; the bottleneck
    width doubles after each stage. Global average pool, linear head."""

    def __init__(self, blocks=(3, 3, 3), cardinality=32, bottleneck_width=4, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True)
        )
        stages, in_ch, width = [], 64, bottleneck_width
        for i, n in enumerate(blocks):
            layers = []
            for j in range(n):
                stride = 2 if (i > 0 and j == 0) else 1
                layers.append(ResNeXtBlock(in_ch, cardinality, width, stride))
                in_ch = ResNeXtBlock.expansion * cardinality * width
            stages.append(nn.Sequential(*layers))
            width *= 2
        self.stages = nn.Sequential(*stages)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, x):
        x = self.stages(self.stem(x))
        return self.fc(torch.flatten(F.adaptive_avg_pool2d(x, 1), 1))


ARCH_DEFAULTS = {"implementation_version": 1}  # scheme G has no architecture switches


def architecture_from_config():
    return {}


def validate_architecture(options):
    pass  # resolve_architecture already rejects unknown options and other implementation versions


def describe_architecture(options):
    return MODEL_DESCRIPTION


def architecture_feature(options):
    return None


def expected_param_count(options):
    return EXPECTED_PARAMS


def architecture_topology(options):
    """Stem width, output channels and number of blocks of each stage (saved with the architecture options)."""
    return {"stem_channels": 64, "stage_channels": [256, 512, 1024], "blocks": [3, 3, 3]}


def make_model(num_classes=10, architecture=None):
    resolve_architecture(architecture)
    return ResNeXt((3, 3, 3), cardinality=32, bottleneck_width=4, num_classes=num_classes)''',
r'''PARAM_BUDGET = 5_000_000


def init_weights(model):
    """Kaiming-normal (fan_out) convs and unit/zero BN, as in torchvision's ResNet."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def count_params(model):
    """Number of trainable parameters (conv/linear weights and biases, BN gamma/beta)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


ARCHITECTURE_IMPLEMENTATION_VERSION = 1  # Bump when the meaning of a switch/forward pass changes.


def resolve_architecture(options=None):
    """None reads UI switches; an explicit dict (including {}) uses saved/baseline options."""
    options = architecture_from_config() if options is None else options
    if not isinstance(options, dict):
        raise ValueError("Architecture options must be a dictionary")
    unknown = set(options) - set(ARCH_DEFAULTS)
    if unknown:
        raise ValueError(f"Unknown architecture options for model {MODEL}: {sorted(unknown)}")
    resolved = dict(ARCH_DEFAULTS, **options)
    version = resolved["implementation_version"]
    if type(version) is not int or version != ARCHITECTURE_IMPLEMENTATION_VERSION:
        raise ValueError("Unsupported architecture implementation_version; use the matching notebook code")
    validate_architecture(resolved)
    return resolved


def saved_architecture(args):
    """Missing legacy fields mean baseline; malformed saved options must never read UI globals."""
    options = args.get("architecture", ARCH_DEFAULTS)
    if not isinstance(options, dict):
        raise ValueError("Saved architecture options must be a dictionary")
    return resolve_architecture(options)


def architecture_details(options):
    """Resolved topology accompanies options in configs, checkpoints and evaluation reports."""
    details = resolve_architecture(options)
    if MODEL not in ("A", "B", "C"):  # E/F/G define architecture_topology next to their model (section 4)
        return dict(details, model=MODEL, **architecture_topology(details), num_classes=NUM_CLASSES)
    channels = WIDTH_PRESETS[details["width_preset"]] if MODEL == "B" else (64, 128, 256)
    return dict(details, model=MODEL, stem_channels=16 if MODEL == "B" else 64,
                stage_channels=list(channels), blocks=[3, 3, 3] if MODEL == "B" else [3, 5, 3],
                num_classes=NUM_CLASSES)


def build_model(num_classes=NUM_CLASSES, architecture=None):
    """Build this notebook's model, check the 5M budget, and return (model, n_trainable_params)."""
    options = resolve_architecture(architecture)
    model = init_weights(make_model(num_classes, architecture=options))
    # Plain attributes do not alter legacy state_dict keys or baseline repr/fingerprints.
    model._project_architecture = options
    model._project_description = describe_architecture(options)
    n_params = count_params(model)
    assert n_params <= PARAM_BUDGET, f"model {MODEL} has {n_params:,} trainable params > {PARAM_BUDGET:,}"
    return model, n_params


_model, N_PARAMS = build_model()
print(f"Model {MODEL}: {_model._project_description}")
print(f"Architecture options: {_model._project_architecture}")
print(f"Trainable parameters: {N_PARAMS:,} (budget {PARAM_BUDGET:,})")
assert N_PARAMS <= PARAM_BUDGET
assert N_PARAMS == expected_param_count(_model._project_architecture), "Unexpected architecture parameter count"
if SHOW_SUMMARY:
    try:
        from torchinfo import summary
    except ImportError:
        print("torchinfo is not installed; layer summary skipped (the parameter count above is complete)")
    else:
        print(summary(_model, input_size=(1, 3, 32, 32), depth=3, col_names=("output_size", "num_params"),
                      device="cpu", verbose=0))
del _model''',
)

In [ ]:
# ---- one namespace per model family, as if that training notebook's section 4 had run
import contextlib
import types

FAMILY_NOTEBOOKS = {"A": "A_resnet.ipynb", "B": "B_wrn.ipynb", "C": "C_seresnet.ipynb",
                    "E": "E_pyramiad_HW.ipynb", "F": "F_shake.ipynb", "G": "G_resnext.ipynb"}
# Baseline switch values, used only by each copied cell's own baseline build; checkpoints always rebuild from
# their saved architecture options (saved_architecture), whatever these values are.
FAMILY_SWITCH_DEFAULTS = {"A": {"CHANNEL_ATTENTION": "none", "ECA_KERNEL_SIZE": 3, "DEPTH_PRESET": "baseline",
                                "CAPACITY_PRESET": "baseline", "ACTIVATION_ORDER": "post"},
                          "B": {"WIDTH_PRESET": "baseline"},
                          "C": {"SE_REDUCTION": 16, "IBN_STAGE1": False},
                          "E": {"DROP_PATH": 0.0, "HEAD": "baseline"}, "F": {}, "G": {}}
# Copied helpers whose global names (MODEL, build_model, ...) must resolve inside the family namespace.
FAMILY_BOUND_HELPERS = ("model_fingerprint", "validate_checkpoint", "load_model")
_FAMILY_NAMESPACES = {}


def _rebind(function, namespace):
    bound = types.FunctionType(function.__code__, namespace, function.__name__, function.__defaults__,
                               function.__closure__)
    bound.__kwdefaults__ = function.__kwdefaults__
    bound.__doc__ = function.__doc__
    return bound


def family_namespace(model):
    """Execute the verbatim copy of `model`'s section 4 in a private namespace (cached)."""
    if model in _FAMILY_NAMESPACES:
        return _FAMILY_NAMESPACES[model]
    if model not in MODEL_SOURCES:
        raise ValueError(f"Unknown model family {model!r}; expected one of {sorted(MODEL_SOURCES)}")
    shared = {k: v for k, v in globals().items() if not k.startswith("_")}
    ns = dict(shared, __name__=f"q08_family_{model}", SHOW_SUMMARY=False, **FAMILY_SWITCH_DEFAULTS[model])
    with contextlib.redirect_stdout(io.StringIO()) as printed:
        for index, source in enumerate(MODEL_SOURCES[model]):
            exec(compile(source, f"{FAMILY_NOTEBOOKS[model]}:section4-cell{index + 1}", "exec"), ns)
    for name in FAMILY_BOUND_HELPERS:
        ns[name] = _rebind(globals()[name], ns)
    if ns["MODEL"] != model:
        raise RuntimeError(f"MODEL_SOURCES[{model!r}] defines model {ns['MODEL']!r}")
    summary = [line for line in printed.getvalue().splitlines() if line.startswith("Model ")]
    print(f"[family {model}] {FAMILY_NOTEBOOKS[model]} section 4 loaded; baseline build "
          f"{ns['N_PARAMS']:,} parameters" + (f" ({summary[0]})" if summary else ""))
    _FAMILY_NAMESPACES[model] = ns
    return ns

## 6. Evaluate checkpoints on clean and shifted validation (Part 2)

For every run in `Q08_RUNS` and each saved checkpoint role: the dataset fingerprint, model identity, resolved
architecture, model fingerprint, validation indices and parameter count are checked first. Clean accuracy is then
computed and compared with every stored value for the same epoch and weights (the checkpoint's training-log row,
the `*-validation.json` exports and the prediction report); a difference above `Q08_REPRO_TOLERANCE_PP` stops the
notebook before the shifted pass. All 39 sets are evaluated in one batched pass per weights and flip direction.

Reported per protocol: accuracy and cross-entropy per corruption × severity, per-corruption, per-severity and
per-family means, `q08_score_pct` (family-balanced mean), the plain mean over all sets, the worst family, per-class
recall on clean validation and averaged over the shifted sets (with the drop per class), drops versus clean, and
`q08_score_excluding_overlap_pct`, which drops the families that the run's own training augmentation already
contains (AutoAugment: photometric and sharpness). Unknown augmentation settings are flagged instead of guessed.
`is_primary` marks the pre-declared last/EMA/flip protocol, `is_submission` the protocol of a saved submission CSV.
Per-image softmax probabilities (float16, clean first, then the 39 sets) are saved in one compressed `.npz` per run
beside the JSON/CSV, by default only for the primary protocol and the submission protocol (about 4 MB each before
compression); `Q08_SAVE_ALL_PROBABILITIES=True` saves all eight. Section 8 reuses them. Checkpoints, stored
validation exports and prediction reports come from the first searched root that has them (`RESULT_DIR`, then
`Q08_EXTRA_RESULT_ROOTS`); `sources` in each report records which. Outputs are written under `RESULT_DIR` only. A
listed run found in no root is skipped with a message; identity, split or reproduction failures still stop the
notebook.
Q08 numbers are a proxy for robustness to generic shifts, not an estimate of the Kaggle score.

In [ ]:
# ---- Part 2 helpers: saved checkpoints on clean + shifted validation
Q08_ROLE = f"q08-shiftval-v{Q08_SET_VERSION}"
Q08_SEVERITY_BANDS = {1: (1.0, 3.0, "light"), 2: (3.0, 8.0, "medium"), 3: (8.0, 20.0, "heavy")}  # drop in points
# Training augmentations whose operations overlap a Q08 family. A run's accuracy on an overlapping family is
# partly in-distribution for that run; compare runs with q08_score_excluding_overlap_pct as well.
AUGMENTATION_OVERLAP = {
    "basic": (),
    "cutout": (),
    # torchvision AutoAugment (CIFAR10 policy): Brightness, Contrast, Color, AutoContrast, Equalize, Invert,
    # Solarize, Posterize, and Sharpness with factors 0.1-1.9 (smoothing and sharpening).
    "autoaug": ("photometric", "sharpness"),
}
_AUGMENTATION_WORDS = ("degrad", "blur", "noise", "jpeg", "jitter", "gamma", "bright", "contrast", "satur",
                       "sharp", "pixel", "randaug", "trivial", "color", "colour", "photometric", "corrupt")


def _active(value):
    return value not in (None, False, 0, 0.0, "", "none", "off", [], {})


def augmentation_overlap(args):
    """Q08 families that a run's training augmentation already contains, plus warnings for unknown recipes."""
    aug = args.get("aug")
    families = set(AUGMENTATION_OVERLAP.get(aug, ()))
    unknown = []
    if aug not in AUGMENTATION_OVERLAP:
        unknown.append(f"unrecognised AUG={aug!r}")
    flagged = sorted(k for k, v in args.items() if any(w in k.lower() for w in _AUGMENTATION_WORDS) and _active(v))
    if flagged:
        unknown.append(f"augmentation-like settings {flagged}")
    return {"families": [f for f in Q08_FAMILIES if f in families], "unknown": unknown,
            "note": ("overlap unknown: check the recipe before discounting families" if unknown else
                     "AutoAugment includes brightness/contrast/colour/sharpness-type operations" if families else
                     "no training augmentation overlaps a Q08 family"),
            "recipe": {k: args.get(k) for k in ("aug", "autoaug_erase", "cutout_size", "crop_padding_mode",
                                                "mixup", "label_smoothing", "epochs")}}


def _artifact_folders(result_root, run_id, category):
    """Canonical and legacy-alias folders of one category (older Drive copies may keep a legacy run ID).

    `result_root` may be a list of roots in search order: the folders of the first root that has any are returned.
    """
    if isinstance(result_root, (list, tuple)):
        return next((found for root in result_root if (found := _artifact_folders(root, run_id, category))), [])
    canonical = canonical_run_id(run_id)
    names = [canonical] + [legacy for legacy, target in RUN_ID_ALIASES.items() if target == canonical]
    parent = getattr(run_paths(result_root, canonical), category).parent
    return [parent / name for name in names if (parent / name).is_dir()]


def _read_json(path):
    try:
        return json.loads(Path(path).read_text())
    except (OSError, ValueError):
        print(f"Warning: unreadable {path}; ignored")
        return None


def _number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)


def recorded_clean_accuracies(result_root, run_id, ckpt):
    """Stored clean validation accuracies for this checkpoint's epoch (training row, exports, prediction reports)."""
    refs = []
    metrics = ckpt.get("metrics") or {}
    for weights, key in (("model", "val_acc"), ("ema", "ema_val_acc")):
        if _number(metrics.get(key)):
            refs.append({"source": f"checkpoint metrics (epoch {ckpt['epoch']})", "weights": weights,
                         "tta": False, "accuracy_pct": float(metrics[key])})
    sources = [(path, "metrics") for folder in _artifact_folders(result_root, run_id, "validation")
               for path in sorted(folder.glob("*-validation.json"))]
    sources += [(path, "val") for folder in _artifact_folders(result_root, run_id, "test")
                for path in sorted(folder.glob("*_prediction-report*.json"))]
    for path, field in sources:
        data = _read_json(path)
        if not data or data.get("epoch") != ckpt["epoch"] or data.get("cache_fingerprint", ckpt["cache_fingerprint"]) != ckpt["cache_fingerprint"]:
            continue
        for weights, values in (data.get(field) or {}).items():
            for mode, tta in (("no_tta", False), ("flip_tta", True)):
                if isinstance(values, dict) and _number(values.get(mode)):
                    refs.append({"source": path.name, "weights": weights, "tta": tta, "accuracy_pct": float(values[mode])})
    return refs


def recorded_submissions(result_root, run_id):
    """Checkpoint/weights/TTA of every prediction report (submission CSV) saved for this run."""
    found = []
    for folder in _artifact_folders(result_root, run_id, "test"):
        for path in sorted(folder.glob("*_prediction-report*.json")):
            data = _read_json(path)
            if not data:
                continue
            checkpoint = str(data.get("checkpoint", ""))
            role = ("last" if checkpoint.endswith("_last-ckpt.pt") else
                    "best" if checkpoint.endswith("_best-ckpt.pt") else None)
            found.append({"report": path.name, "checkpoint_role": role, "epoch": data.get("epoch"),
                          "weights": data.get("weights"), "tta": data.get("tta"),
                          "submission": Path(str(data.get("submission", ""))).name})
    return found


def q08_check_reproduction(label, weights, clean_metrics, references):
    """Clean accuracy must reproduce every stored value for the same epoch/weights within the tolerance."""
    rows = [dict(ref, q08_accuracy_pct=clean_metrics[ref["tta"]]["accuracy_pct"],
                 difference_pp=clean_metrics[ref["tta"]]["accuracy_pct"] - ref["accuracy_pct"])
            for ref in references if ref["weights"] == weights]
    worst = max((abs(r["difference_pp"]) for r in rows), default=None)
    if Q08_SMOKE:
        status = "not checked: smoke subset"
    elif not rows:
        status = "no stored reference"
        print(f"Warning: {label}: no stored clean accuracy for this epoch/weights; reproduction unchecked")
    elif worst > Q08_REPRO_TOLERANCE_PP + 1e-9:
        details = "; ".join(f"{r['source']} {'flip' if r['tta'] else 'plain'} {r['accuracy_pct']:.2f}% vs "
                            f"{r['q08_accuracy_pct']:.2f}%" for r in rows)
        raise RuntimeError(f"{label}: clean validation does not reproduce the stored accuracy within "
                           f"{Q08_REPRO_TOLERANCE_PP}pp ({details}); check the checkpoint, split and model code")
    else:
        status = "reproduced"
    return {"status": status, "max_abs_difference_pp": worst, "tolerance_pp": Q08_REPRO_TOLERANCE_PP,
            "references": rows}


def _set_metrics(probs, labels):
    metrics = classification_metrics(probs, labels)
    metrics.pop("confusion_matrix")
    return metrics


def _per_class_drop(clean_per_class, sets):
    """Per-class recall (percent) on clean validation and averaged over every shifted set, and the drop in points."""
    if not clean_per_class or not all("per_class_recall" in m for m in sets.values()):
        return None
    clean, shifted, drop = {}, {}, {}
    for name, value in clean_per_class.items():
        values = [m["per_class_recall"][name] for m in sets.values()]
        clean[name] = None if value is None else 100.0 * value
        shifted[name] = None if value is None or None in values else 100.0 * float(np.mean(values))
        drop[name] = None if shifted[name] is None else clean[name] - shifted[name]
    return {"clean_recall_pct": clean, "mean_shifted_recall_pct": shifted, "drop_pp": drop}


def q08_summarize(clean_accuracy, sets, overlap_families, clean_per_class=None):
    """Per-corruption, per-severity and per-family means; the headline q08_score weights the four families equally.

    Also the worst family (lowest family mean) and, given the clean per-class recall, each class's mean recall drop.
    """
    acc = {key: value["accuracy_pct"] for key, value in sets.items()}
    family_of = {key: Q08_CORRUPTIONS[q08_key_info(key)[0]]["family"] for key in acc}
    family = {f: float(np.mean([a for k, a in acc.items() if family_of[k] == f])) for f in Q08_FAMILIES}
    score = float(np.mean(list(family.values())))
    kept = [f for f in Q08_FAMILIES if f not in overlap_families]
    score_kept = float(np.mean([family[f] for f in kept])) if kept else None
    mean_all = float(np.mean(list(acc.values())))
    worst = min(Q08_FAMILIES, key=lambda f: family[f])
    return {"clean_accuracy_pct": clean_accuracy,
            "q08_score_pct": score, "q08_drop_pp": clean_accuracy - score,
            "mean_corruption_accuracy_pct": mean_all, "mean_drop_pp": clean_accuracy - mean_all,
            "worst_family": worst, "worst_family_accuracy_pct": family[worst],
            "worst_family_drop_pp": clean_accuracy - family[worst],
            "family_accuracy_pct": family, "family_drop_pp": {f: clean_accuracy - a for f, a in family.items()},
            "corruption_accuracy_pct": {name: float(np.mean([acc[q08_set_key(name, s)] for s in Q08_SEVERITIES]))
                                        for name in Q08_CORRUPTIONS},
            "severity_accuracy_pct": {str(s): float(np.mean([acc[q08_set_key(n, s)] for n in Q08_CORRUPTIONS]))
                                      for s in Q08_SEVERITIES},
            "q08_score_excluding_overlap_pct": score_kept, "excluded_overlap_families": list(overlap_families),
            "set_drop_pp": {key: clean_accuracy - a for key, a in acc.items()},
            "per_class": _per_class_drop(clean_per_class, sets)}


class Q08MissingRun(FileNotFoundError):
    """None of a run's requested checkpoints exists under any searched result root (the run is skipped, not fatal)."""


def q08_protocol_key(checkpoint, weights, tta):
    return f"{checkpoint}_{weights}_{'flip_tta' if tta else 'no_tta'}"


def q08_locate_run(run_id, checkpoints, roots):
    """(root, {role: path}) of the first root, in search order, holding any requested checkpoint of the run."""
    for root in roots:
        folder = run_paths(Path(root), run_id).checkpoints
        paths = {role: checkpoint_path(folder, role) for role in checkpoints}
        if any(path.is_file() for path in paths.values()):
            return Path(root), paths
    searched = ", ".join(str(run_paths(Path(root), run_id).checkpoints) for root in roots)
    raise Q08MissingRun(f"{run_id}: no {'/'.join(checkpoints)} checkpoint under {searched} (legacy names checked "
                        "too); add the result folder that holds it to Q08_EXTRA_RESULT_ROOTS or copy its checkpoint "
                        "folder to RESULT_DIR to evaluate it")


def q08_evaluate_run(run_id, device=None, checkpoints=None, weight_choices=None, batch_size=None, result_root=None):
    """Evaluate one run's saved checkpoints x weights x plain/flip TTA on clean and every Q08 set.

    Checkpoints and stored metadata are searched in RESULT_DIR, then the extra roots (`result_root` restricts the
    search to one folder). Returns (report, probabilities): probabilities[protocol key] is float16
    (1 + n_sets, N, 10), clean first.
    """
    device = device or DEV
    checkpoints = tuple(checkpoints or Q08_CHECKPOINTS)
    weight_choices = tuple(weight_choices or Q08_WEIGHTS)
    batch_size = batch_size or EVAL_BATCH_SIZE
    roots = [Path(result_root)] if result_root is not None else q08_result_roots()
    run_id = canonical_run_id(run_id)
    for role in checkpoints:
        if role not in ("best", "last"):
            raise ValueError("Q08_CHECKPOINTS entries must be best or last")
    checkpoint_root, paths = q08_locate_run(run_id, checkpoints, roots)
    submissions = recorded_submissions(roots, run_id)
    metadata_root = {category: next((str(r) for r in roots if _artifact_folders(r, run_id, category)), None)
                     for category in ("validation", "test")}
    shifted = Q08_STACK.reshape((-1,) + VAL_X.shape[1:])  # (n_sets * N, 32, 32, 3) view, one batched pass
    n_sets, n = len(Q08_KEYS), len(VAL_Y)
    report = {"run_id": run_id, "q08_set": Q08_SET_NAME, "q08_set_version": Q08_SET_VERSION,
              "q08_set_sha256": Q08_SET_SHA256, "q08_set_status": Q08_STATUS, "q08_set_source": Q08_SOURCE_USED,
              "smoke_subset": Q08_N if Q08_SMOKE else None, "n_images_per_set": n, "set_keys": list(Q08_KEYS),
              "submissions": submissions, "protocols": [],
              "sources": {"checkpoint_root": str(checkpoint_root), "checkpoint_root_kind": q08_root_kind(checkpoint_root),
                          "validation_exports_root": metadata_root["validation"],
                          "prediction_reports_root": metadata_root["test"],
                          "searched_roots": [str(r) for r in roots], "outputs_root": str(PATHS.result_dir)},
              "inference": {"precision": "fp32", "channels_last": False, "batch_size": batch_size,
                            "tta": "mean of softmax(original) and softmax(horizontal flip)",
                            "device": device_name(device), "torch": torch.__version__},
              "primary_rule": "last checkpoint, EMA weights when saved (else raw), flip TTA"}
    probabilities = {}
    t0 = time.perf_counter()
    for role in checkpoints:
        path = paths[role]
        if not path.is_file():
            print(f"Warning: {run_id}: no {role} checkpoint at {path}; skipped")
            continue
        ckpt = load_checkpoint(path)
        fam = family_namespace(ckpt["args"]["model"])
        fam["validate_checkpoint"](ckpt, META["fingerprint"], run_id=run_id)
        if not np.array_equal(ckpt["val_idx"].numpy(), VAL_IDX):
            raise ValueError(f"{run_id}: checkpoint validation indices differ from the Q08 split")
        options = fam["saved_architecture"](ckpt["args"])
        expected_params = fam["expected_param_count"](options)
        overlap = augmentation_overlap(ckpt["args"])
        report.update(model=ckpt["args"]["model"], architecture=fam["architecture_details"](options),
                      model_description=fam["describe_architecture"](options), training_augmentation=overlap)
        mean, std = ckpt["norm"]["mean"], ckpt["norm"]["std"]
        references = recorded_clean_accuracies(roots, run_id, ckpt)
        primary_weights = "ema" if ckpt.get("ema") is not None else "model"
        checkpoint_sha256 = sha256_file(path)
        for weights in weight_choices:
            if weights not in ("model", "ema"):
                raise ValueError("Q08_WEIGHTS entries must be model or ema")
            if weights == "ema" and ckpt.get("ema") is None:
                continue
            model, n_params = fam["load_model"](ckpt, weights, device)
            if n_params != ckpt.get("n_params", n_params) or n_params != expected_params or n_params > fam["PARAM_BUDGET"]:
                raise ValueError(f"{run_id}: parameter count {n_params:,} differs from the saved/expected count")
            label = f"{run_id} {role} epoch {ckpt['epoch']} {weights}"
            # Clean first: stop before the long pass if the stored clean accuracy is not reproduced.
            plain = F.softmax(predict_logits(model, VAL_X, mean, std, device, batch_size), 1).numpy()
            flipped = F.softmax(predict_logits(model, VAL_X, mean, std, device, batch_size, flip=True), 1).numpy()
            clean = {False: classification_metrics(plain, VAL_Y), True: classification_metrics(0.5 * (plain + flipped), VAL_Y)}
            reproduction = q08_check_reproduction(label, weights, clean, references)
            s_plain = F.softmax(predict_logits(model, shifted, mean, std, device, batch_size), 1).numpy()
            s_flip = F.softmax(predict_logits(model, shifted, mean, std, device, batch_size, flip=True), 1).numpy()
            for tta in (False, True):
                probs = (0.5 * (s_plain + s_flip) if tta else s_plain).reshape(n_sets, n, NUM_CLASSES)
                sets = {key: _set_metrics(probs[i], VAL_Y) for i, key in enumerate(Q08_KEYS)}
                key = q08_protocol_key(role, weights, tta)
                clean_probs = 0.5 * (plain + flipped) if tta else plain
                probabilities[key] = np.concatenate([clean_probs[None], probs]).astype(np.float16)
                is_submission = any(s["epoch"] == ckpt["epoch"] and s["weights"] == weights and bool(s["tta"]) == tta
                                    and s["checkpoint_role"] in (None, role) for s in submissions)
                report["protocols"].append({
                    "probabilities_key": key, "checkpoint": role, "checkpoint_file": path.name, "checkpoint_sha256": checkpoint_sha256,
                    "epoch": ckpt["epoch"], "planned_epochs": ckpt["args"]["epochs"], "weights": weights, "tta": tta,
                    "is_primary": role == "last" and weights == primary_weights and tta,
                    "is_submission": is_submission, "n_params": n_params,
                    "clean": clean[tta], "clean_reproduction": reproduction, "sets": sets,
                    "summary": q08_summarize(clean[tta]["accuracy_pct"], sets, overlap["families"],
                                             clean[tta]["per_class_recall"])})
            print(f"  {label}: clean plain {clean[False]['accuracy_pct']:.2f}% / flip {clean[True]['accuracy_pct']:.2f}% "
                  f"({reproduction['status']}) | q08 flip {report['protocols'][-1]['summary']['q08_score_pct']:.2f}%")
            del model
            if device.type == "cuda":
                torch.cuda.empty_cache()
    report["elapsed_s"] = round(time.perf_counter() - t0, 1)
    report["created_at"] = datetime.now(ZoneInfo("America/New_York")).isoformat(timespec="seconds")
    return report, probabilities


def q08_long_rows(report):
    rows = []
    overlap = set(report["training_augmentation"]["families"])
    for p in report["protocols"]:
        base = {"run_id": report["run_id"], "model": report["model"], "checkpoint": p["checkpoint"],
                "epoch": p["epoch"], "weights": p["weights"], "tta": p["tta"], "is_primary": p["is_primary"],
                "is_submission": p["is_submission"], "q08_set": report["q08_set"], "q08_set_sha256": report["q08_set_sha256"]}
        rows.append(dict(base, set="clean", corruption="clean", family="clean", severity=0, params=None,
                         accuracy_pct=p["clean"]["accuracy_pct"], correct=p["clean"]["correct"], n=p["clean"]["n"],
                         cross_entropy=p["clean"]["cross_entropy"], drop_pp=0.0, training_overlap=False))
        for key, m in p["sets"].items():
            name, severity, spec = q08_key_info(key)
            rows.append(dict(base, set=key, corruption=name, family=spec["family"], severity=severity,
                             params=json.dumps(spec["params"][severity - 1]), accuracy_pct=m["accuracy_pct"],
                             correct=m["correct"], n=m["n"], cross_entropy=m["cross_entropy"],
                             drop_pp=p["summary"]["set_drop_pp"][key], training_overlap=spec["family"] in overlap))
    return rows


def q08_probability_keys(report):
    """Protocols whose per-image probabilities are saved: the primary and submission ones unless all are requested."""
    keys = [p["probabilities_key"] for p in report["protocols"]]
    if Q08_SAVE_ALL_PROBABILITIES:
        return keys
    chosen = list(dict.fromkeys(p["probabilities_key"] for p in report["protocols"] if p["is_primary"] or p["is_submission"]))
    if not chosen:
        print(f"Warning: {report['run_id']}: neither the primary nor a submission protocol was evaluated; "
              "saving the probabilities of all evaluated protocols")
    return chosen or keys


def q08_write_run_report(report, probabilities, result_root=None):
    """Per-run JSON (all protocols), long-format CSV and per-image probabilities (.npz, float16; primary and
    submission protocols by default) in evaluations/validation/<RUN_ID>/, all with the same dated stem."""
    artifacts = run_paths(Path(result_root or PATHS.result_dir), report["run_id"])
    artifacts.validation.mkdir(parents=True, exist_ok=True)
    role = Q08_ROLE + ("-smoke" if Q08_SMOKE else "")
    path = artifact_path(artifacts, "validation", role, ".json", event=True, unique=True)
    saved = q08_probability_keys(report)
    for protocol in report["protocols"]:
        protocol["probabilities_saved"] = protocol["probabilities_key"] in saved
    report["probabilities_saved"] = saved
    write_json_atomic(report, path)
    pd.DataFrame(q08_long_rows(report)).to_csv(path.with_suffix(".csv"), index=False)
    meta = {"run_id": report["run_id"], "q08_set": Q08_SET_NAME, "q08_set_sha256": Q08_SET_SHA256,
            "smoke_subset": report["smoke_subset"], "set_keys": ["clean"] + list(Q08_KEYS),
            "layout": "probabilities[protocol] has shape (1 + n_sets, N, 10): row 0 clean, then set_keys[1:]",
            "protocols": [{k: p[k] for k in ("probabilities_key", "checkpoint", "epoch", "weights", "tta",
                                            "is_primary", "is_submission")} for p in report["protocols"]
                          if p["probabilities_saved"]]}
    partial = path.with_name(f".{path.stem}.npz.{time.time_ns()}.tmp")
    try:
        with open(partial, "wb") as f:
            np.savez_compressed(f, labels=np.asarray(VAL_Y, dtype="<i8"), val_indices=np.asarray(VAL_IDX[:len(VAL_Y)], dtype="<i8"),
                                meta_json=np.array(json.dumps(meta, sort_keys=True)), **{k: probabilities[k] for k in saved})
        replace_file(partial, path.with_suffix(".npz"))
    finally:
        partial.unlink(missing_ok=True)
    return path


def q08_summary_rows(reports):
    rows = []
    for report in reports:
        overlap = report["training_augmentation"]
        for p in report["protocols"]:
            s = p["summary"]
            rows.append({"run_id": report["run_id"], "model": report["model"], "checkpoint": p["checkpoint"],
                         "epoch": p["epoch"], "weights": p["weights"], "tta": p["tta"], "is_primary": p["is_primary"],
                         "is_submission": p["is_submission"], "clean_pct": s["clean_accuracy_pct"],
                         "q08_score_pct": s["q08_score_pct"], "q08_drop_pp": s["q08_drop_pp"],
                         "mean_corruption_pct": s["mean_corruption_accuracy_pct"],
                         "worst_family": s["worst_family"], "worst_family_pct": s["worst_family_accuracy_pct"],
                         **{f"{f}_pct": s["family_accuracy_pct"][f] for f in Q08_FAMILIES},
                         **{f"recall_drop_{c}_pp": (s["per_class"] or {}).get("drop_pp", {}).get(c) for c in CLASSES},
                         "q08_score_excluding_overlap_pct": s["q08_score_excluding_overlap_pct"],
                         "overlap_families": "+".join(overlap["families"]) or "none",
                         "overlap_unknown": "; ".join(overlap["unknown"]) or "",
                         "clean_reproduction": p["clean_reproduction"]["status"],
                         "q08_set": report["q08_set"], "q08_set_sha256": report["q08_set_sha256"],
                         "q08_set_status": report["q08_set_status"],
                         "checkpoint_root_kind": report["sources"]["checkpoint_root_kind"],
                         "checkpoint_root": report["sources"]["checkpoint_root"]})
    return rows


def q08_severity_check(report, checkpoint=None, weights=None, tta=True):
    """One-time check: per-corruption drops of one reference run against the light/medium/heavy bands."""
    candidates = [p for p in report["protocols"] if (checkpoint is None or p["checkpoint"] == checkpoint)
                  and (weights is None or p["weights"] == weights) and p["tta"] == tta]
    if not candidates:
        raise ValueError("No evaluated protocol matches the requested checkpoint/weights/TTA")
    protocol = next((p for p in candidates if p["is_primary"]), candidates[0])
    drops = protocol["summary"]["set_drop_pp"]
    rows = []
    for name, spec in Q08_CORRUPTIONS.items():
        source = spec["source"]
        row = {"corruption": name, "family": spec["family"],
               "source": "CIFAR-10-C" if source.startswith("CIFAR-10-C") else "defined" if source.startswith("defined") else "mixed"}
        for s in Q08_SEVERITIES:
            lo, hi, band = Q08_SEVERITY_BANDS[s]
            d = drops[q08_set_key(name, s)]
            row[f"s{s}_drop_pp"] = round(d, 2)
            row[f"s{s}_{band}"] = "below" if d < lo else "above" if d > hi else "in band"
        rows.append(row)
    table = pd.DataFrame(rows)
    print(f"Severity check: {report['run_id']} | {protocol['checkpoint']} epoch {protocol['epoch']} | "
          f"{protocol['weights']} | {'flip' if protocol['tta'] else 'plain'} | clean {protocol['clean']['accuracy_pct']:.2f}% | "
          f"{Q08_SET_NAME} ({Q08_STATUS})")
    print("Target bands (drop vs clean): " + ", ".join(f"s{s} {b} {lo:g}-{hi:g}pp" for s, (lo, hi, b) in Q08_SEVERITY_BANDS.items()))
    print(table.to_string(index=False))
    return table, protocol

In [ ]:
if DEV.type == "cuda":
    torch.backends.cudnn.benchmark = bool(HW.cudnn_benchmark)
Q08_REPORTS, Q08_SKIPPED_RUNS = {}, {}
_roots = q08_result_roots() if Q08_RUNS else []
if _roots:
    print("[q08] checkpoint search order: " + " -> ".join(f"{r} ({q08_root_kind(r)})" for r in _roots)
          + f"; outputs go to {PATHS.result_dir} only")
for _run in Q08_RUNS:
    print(f"[q08] evaluating {_run}", flush=True)
    try:
        _report, _probabilities = q08_evaluate_run(_run)
    except Q08MissingRun as _missing:  # other errors (identity, split, reproduction) still stop the notebook
        Q08_SKIPPED_RUNS[canonical_run_id(_run)] = str(_missing)
        print(f"[q08] SKIPPED {_missing}", flush=True)
        continue
    _path = q08_write_run_report(_report, _probabilities)
    Q08_REPORTS[_report["run_id"]] = _report
    print(f"[q08] {_report['run_id']}: checkpoints from {_report['sources']['checkpoint_root']} "
          f"({_report['sources']['checkpoint_root_kind']}); {len(_report['protocols'])} protocols in {_report['elapsed_s']:.0f}s -> "
          f"{_path.with_suffix('')}.json/.csv/.npz (probabilities: {', '.join(_report['probabilities_saved'])})")
if Q08_SKIPPED_RUNS:
    print(f"[q08] {len(Q08_SKIPPED_RUNS)} of {len(Q08_RUNS)} runs skipped (checkpoints in no searched result root): "
          + ", ".join(Q08_SKIPPED_RUNS))
if Q08_REPORTS:
    Q08_SUMMARY = pd.DataFrame(q08_summary_rows(Q08_REPORTS.values()))
    _role = "summary-smoke" if Q08_SMOKE else "summary"
    Q08_SUMMARY_PATH = q08_event_path(Path(PATHS.result_dir) / "reports", _role, ".csv")
    Q08_SUMMARY.to_csv(Q08_SUMMARY_PATH, index=False)
    print(f"[q08] summary of {len(Q08_SUMMARY)} protocols: {Q08_SUMMARY_PATH}")
    _shown = Q08_SUMMARY[Q08_SUMMARY["is_primary"] | Q08_SUMMARY["is_submission"]]
    _columns = ["run_id", "checkpoint", "epoch", "weights", "tta", "is_primary", "is_submission", "clean_pct",
                "q08_score_pct", "q08_drop_pp", "worst_family", "worst_family_pct"] + [
                f"{f}_pct" for f in Q08_FAMILIES] + ["q08_score_excluding_overlap_pct", "overlap_families"]
    with pd.option_context("display.width", 250, "display.max_columns", None, "display.precision", 2):
        print("Primary (last, EMA if saved, flip) and submitted protocols; all protocols are in the CSV:")
        print(_shown[_columns].to_string(index=False))
else:
    print("[q08] no run was evaluated (Q08_RUNS empty or all skipped): only the shifted set was prepared")

## 7. One-time severity sanity check

For one pre-declared reference run (`Q08_SEVERITY_REFERENCE_RUN`: C ver-2, the C baseline; primary protocol), each
corruption's accuracy drop is printed against the intended bands: severity 1 light (1–3 points), severity 2 medium
(3–8), severity 3 heavy (8–20). The bands only describe the intent; CIFAR-10-C values are kept even when a corruption
is harder or easier than its band (for example, standard CIFAR training is known to be sensitive to additive noise).
If the reference run was not evaluated, the check is skipped rather than run on another model.
Run this once, record the decision, then lock the set with `Q08_LOCKED_SET_SHA256`. Any later change of corruptions or
severities requires a new `Q08_SET_VERSION`; never adjust severities from hidden-test statistics or public scores.

In [ ]:
Q08_SEVERITY_TABLE = None
if Q08_REPORTS:
    _reference_run = canonical_run_id(Q08_SEVERITY_REFERENCE_RUN or next(iter(Q08_REPORTS)))
    if _reference_run not in Q08_REPORTS:
        # The reference is pre-declared: it is not replaced by whichever run happens to be available.
        print(f"Severity check skipped: Q08_SEVERITY_REFERENCE_RUN {_reference_run} was not evaluated in this "
              "session (not listed in Q08_RUNS, or its checkpoints were found in no result root). Evaluate it, then "
              "rerun this section; set it to None only to use the first evaluated run instead.")
    else:
        Q08_SEVERITY_TABLE, _protocol = q08_severity_check(Q08_REPORTS[_reference_run])
        _path = q08_event_path(Path(PATHS.result_dir) / "reports", "severity-check" + ("-smoke" if Q08_SMOKE else ""), ".csv")
        Q08_SEVERITY_TABLE.assign(run_id=_reference_run, checkpoint=_protocol["checkpoint"], epoch=_protocol["epoch"],
                                  weights=_protocol["weights"], tta=_protocol["tta"], q08_set_sha256=Q08_SET_SHA256
                                  ).to_csv(_path, index=False)
        print(f"[q08] severity check saved: {_path}")
        print("One-time check only. Severities are fixed a priori and must not be fitted to hidden-test statistics or "
              "public scores. If you decide to change one, record why, bump Q08_SET_VERSION, regenerate, then lock the "
              "new set with Q08_LOCKED_SET_SHA256.")

## 8. Ensemble diversity (optional)

For the runs in `Q08_ENSEMBLE_RUNS` (default: the runs evaluated above; a list of run IDs reuses their saved
`*_q08-shiftval-v1.npz` probability files, which must belong to the same locked set; they are searched in
`RESULT_DIR`, then in the extra roots, so another account's files are used as they are) and one protocol per run
(`Q08_ENSEMBLE_PROTOCOL`: `"primary"` by default, or `"submission"`; other protocols need
`Q08_SAVE_ALL_PROBABILITIES=True`), this section reports

- pairwise **disagreement**: the fraction of validation images whose predicted classes differ, on clean validation and
  averaged over each Q08 family;
- **ensemble accuracy** of averaged softmax probabilities, for any subset (`q08_ensemble_scores`), on clean
  validation together with the Q08 score;
- a **greedy forward selection**: start from the best single run on clean validation, repeatedly add the run that most
  increases clean ensemble accuracy, report the Q08 score at every step, and stop when no run increases it.

It is meant for choosing distillation teachers (T05). Selecting runs on the same 5k validation images that measure
them is optimistic; treat small gains (a few images) as ties. Test images are never used.

In [ ]:
# ---- ensemble diversity over saved per-image probabilities (validation images only)
def q08_load_probabilities(run_id, result_root=None):
    """Newest saved Q08 probability file of a run that belongs to the current set (same set SHA-256).

    Searched in RESULT_DIR first, then the extra roots, so files written by another account (its own Drive, or
    synced into this repository's result/) are used as they are; `result_root` restricts the search to one folder.
    """
    role = Q08_ROLE + ("-smoke" if Q08_SMOKE else "")
    pattern = re.compile(rf"_{re.escape(role)}(-\d+)?\.npz$")
    roots = [Path(result_root)] if result_root is not None else q08_result_roots()
    for root in roots:
        paths = [p for folder in _artifact_folders(root, run_id, "validation")
                 for p in folder.glob(f"*_{role}*.npz") if pattern.search(p.name)]
        for path in sorted(paths, key=lambda p: (p.stat().st_mtime_ns, p.name), reverse=True):
            with np.load(path, allow_pickle=False) as z:
                meta = json.loads(str(z["meta_json"][()]))
                if meta.get("q08_set_sha256") != Q08_SET_SHA256 or meta.get("set_keys") != ["clean"] + q08_set_keys():
                    continue
                arrays = {p["probabilities_key"]: z[p["probabilities_key"]] for p in meta["protocols"]}
                labels = z["labels"]
            return path, meta, arrays, labels
    raise FileNotFoundError(f"{run_id}: no saved {role} probabilities for set {Q08_SET_SHA256[:12]} in "
                            f"{', '.join(map(str, roots))}; evaluate it first")


def q08_select_protocol(meta, protocol=None):
    protocol = protocol or Q08_ENSEMBLE_PROTOCOL
    entries = meta["protocols"]
    if protocol == "primary":
        chosen = [p for p in entries if p["is_primary"]]
    elif protocol == "submission":
        chosen = sorted((p for p in entries if p["is_submission"]), key=lambda p: (p["checkpoint"] != "best", not p["tta"]))
    else:
        chosen = [p for p in entries if p["probabilities_key"] == protocol]
    if not chosen:
        raise ValueError(f"{meta['run_id']}: no evaluated protocol matches {protocol!r}")
    return chosen[0]["probabilities_key"]


def q08_ensemble_inputs(run_ids, protocol=None, result_root=None):
    """{run_id: float32 (1 + n_sets, N, 10)} for one protocol per run, plus the shared labels."""
    probs, labels, chosen = {}, None, {}
    for run_id in run_ids:
        run_id = canonical_run_id(run_id)
        try:
            path, meta, arrays, run_labels = q08_load_probabilities(run_id, result_root)
        except FileNotFoundError as missing:
            print(f"[q08] ensemble: skipping {missing}")
            continue
        try:
            key = q08_select_protocol(meta, protocol)
        except ValueError as unmatched:
            print(f"[q08] ensemble: skipping {unmatched} (set Q08_SAVE_ALL_PROBABILITIES=True for other protocols)")
            continue
        if labels is not None and not np.array_equal(labels, run_labels):
            raise ValueError(f"{run_id}: validation labels differ between probability files")
        labels = run_labels
        probs[run_id] = arrays[key].astype(np.float32)
        chosen[run_id] = {"protocol": key, "file": path.name, "root": str(path.parents[3])}
    return probs, labels, chosen


def _q08_groups():
    family_of = [Q08_CORRUPTIONS[q08_key_info(key)[0]]["family"] for key in q08_set_keys()]
    return {"clean": [0], **{f: [1 + i for i, fam in enumerate(family_of) if fam == f] for f in Q08_FAMILIES}}


def q08_ensemble_scores(prob_list, labels):
    """Accuracy of the averaged probabilities: clean, per family (mean over its sets) and the Q08 score."""
    accuracy = (np.mean(prob_list, axis=0).argmax(-1) == labels).mean(axis=-1) * 100  # (1 + n_sets,)
    groups = _q08_groups()
    family = {f: float(accuracy[groups[f]].mean()) for f in Q08_FAMILIES}
    return {"clean_pct": float(accuracy[0]), "q08_score_pct": float(np.mean(list(family.values()))),
            "mean_corruption_pct": float(accuracy[1:].mean()), **{f"{f}_pct": a for f, a in family.items()}}


def q08_disagreement(a, b):
    """Fraction of images with different predicted classes: clean and averaged per Q08 family."""
    rate = (a.argmax(-1) != b.argmax(-1)).mean(axis=-1)
    return {group: float(rate[index].mean()) for group, index in _q08_groups().items()}


def q08_greedy_selection(probs, labels):
    """Forward selection on clean ensemble accuracy (ties: higher Q08 score, then list order); stops without gain."""
    order = list(probs)
    singles = {r: q08_ensemble_scores([probs[r]], labels) for r in order}
    first = max(order, key=lambda r: (singles[r]["clean_pct"], singles[r]["q08_score_pct"], -order.index(r)))
    selected, steps = [first], [dict(step=1, added=first, members=[first], **singles[first])]
    while len(selected) < len(order):
        trials = {r: q08_ensemble_scores([probs[m] for m in selected + [r]], labels) for r in order if r not in selected}
        best = max(trials, key=lambda r: (trials[r]["clean_pct"], trials[r]["q08_score_pct"], -order.index(r)))
        if trials[best]["clean_pct"] <= steps[-1]["clean_pct"] + 1e-9:
            break
        selected.append(best)
        steps.append(dict(step=len(selected), added=best, members=list(selected), **trials[best]))
    return singles, steps


def q08_ensemble_diversity(run_ids, protocol=None, result_root=None):
    """None when fewer than two runs have saved probabilities for the current set."""
    probs, labels, chosen = q08_ensemble_inputs(run_ids, protocol, result_root)
    if len(probs) < 2:
        return None
    order = list(probs)
    pairs = [dict(run_a=a, run_b=b, **{f"disagreement_{g}": v for g, v in q08_disagreement(probs[a], probs[b]).items()})
             for i, a in enumerate(order) for b in order[i + 1:]]
    singles, steps = q08_greedy_selection(probs, labels)
    return {"q08_set": Q08_SET_NAME, "q08_set_sha256": Q08_SET_SHA256, "n_images": int(len(labels)),
            "protocols": chosen, "single_runs": singles, "pairwise_disagreement": pairs,
            "all_runs_ensemble": q08_ensemble_scores([probs[r] for r in order], labels), "greedy_selection": steps,
            "selection_note": "greedy selection uses the same validation images it reports; small gains are ties"}


_ensemble_runs = list(Q08_REPORTS) if Q08_ENSEMBLE_RUNS == "evaluated" else (Q08_ENSEMBLE_RUNS or [])
Q08_ENSEMBLE = q08_ensemble_diversity(_ensemble_runs) if len(_ensemble_runs) >= 2 else None
if Q08_ENSEMBLE is not None:
    _short = {r: canonical_run_id(r).split("_", 1)[-1] for r in Q08_ENSEMBLE["protocols"]}
    with pd.option_context("display.width", 250, "display.max_columns", None, "display.precision", 3):
        print("Single runs (" + ", ".join(f"{_short[r]}: {v['protocol']}" for r, v in Q08_ENSEMBLE["protocols"].items()) + ")")
        print(pd.DataFrame(Q08_ENSEMBLE["single_runs"]).T.rename(index=_short).to_string())
        _pairs = pd.DataFrame(Q08_ENSEMBLE["pairwise_disagreement"]).replace(_short)
        print("Pairwise disagreement (fraction of images):")
        print(_pairs.to_string(index=False))
        _greedy = pd.DataFrame(Q08_ENSEMBLE["greedy_selection"])
        print("Greedy forward selection on clean validation (Q08 score alongside):")
        print(_greedy.assign(added=_greedy["added"].map(_short)).drop(columns="members").to_string(index=False))
        print(f"All {len(_short)} runs averaged: {Q08_ENSEMBLE['all_runs_ensemble']}")
    _role = "ensemble-diversity" + ("-smoke" if Q08_SMOKE else "")
    _path = q08_event_path(Path(PATHS.result_dir) / "reports", _role, ".json")
    write_json_atomic(Q08_ENSEMBLE, _path)
    _pairs.to_csv(_path.with_name(_path.stem + "-pairs.csv"), index=False)
    _greedy.assign(members=_greedy["members"].map("+".join)).to_csv(_path.with_name(_path.stem + "-greedy.csv"), index=False)
    print(f"[q08] ensemble diversity saved: {_path} (+ -pairs.csv, -greedy.csv)")
else:
    print("[q08] ensemble diversity skipped: needs at least two runs with saved probabilities (Q08_ENSEMBLE_RUNS)")